# Phân tích và chuẩn bị dữ liệu biển số

Trong bài thực hành này, chúng ta tìm hiểu dữ liệu cho ba công đoạn: xác định vị trí biển số, tìm vị trí ký tự và nhận dạng từng ký tự. Mỗi phần sẽ đặt câu hỏi, trình bày kết quả bằng bảng hoặc hình, rồi rút ra nhận xét.

**Tiến độ:** đã kiểm tra và tập trung nguồn ở bước 1; đang bổ sung phân tích ảnh YOLO1 ở bước 2. Các phần làm sạch, YOLO2, lọc OCR và xuất dữ liệu huấn luyện sẽ được thực hiện lần lượt. Chưa có số liệu cuối cùng của phương án mới.

## 1. Tìm hiểu các nguồn dữ liệu

Trước khi thống kê, cần biết mỗi bộ được dùng cho công việc nào. YOLO1 và YOLO2 dùng hai bộ riêng. Với OCR, ba nguồn được dùng để tạo train/validation; VNLP được dành riêng để kiểm tra trên một nguồn Việt Nam.

| Công đoạn | Nguồn | Mục đích |
|---|---|---|
| YOLO1 | Global v13 | Xác định vùng biển số trên ảnh |
| YOLO2 | Roboflow OCR v1 | Xác định vị trí ký tự trên ảnh biển số |
| OCR — train/validation | Roboflow, Winter2897, CCPD2019 | Học và lựa chọn mô hình nhận dạng ký tự |
| OCR — test | VNLP | Đánh giá trên nguồn chưa dùng để huấn luyện |

Roboflow phục vụ cả YOLO2 và OCR: ảnh và khung ký tự dùng cho YOLO2; từng ảnh ký tự cắt ra cùng nhãn gốc dùng cho OCR. Chỉ cần lưu một bản nguồn Roboflow.


### 1.1. Kết quả sẽ được lưu ở đâu?

Thư mục chung là **`MyDrive/ML/Do_an_bien_so`**. Tên thư mục thể hiện trực tiếp nội dung để thuận tiện tìm lại.

| Thư mục | Nội dung |
|---|---|
| `01_Du_lieu_goc` | Các nguồn Global v13, Roboflow, Winter, CCPD và VNLP |
| `02_Du_lieu_huan_luyen` | Kết quả chuẩn bị riêng cho YOLO1, YOLO2 và OCR |
| `03_Bao_cao` | Bảng thống kê, biểu đồ và kết luận |
| `04_Anh_dai_dien` | Ảnh mẫu theo nhiệm vụ, nguồn và kích thước |
| `05_Notebook` | Notebook và hướng dẫn |
| `06_Ket_qua_cu` | Các kết quả trước đây để đối chiếu |

Các tệp nguồn đã được tập trung vào `01_Du_lieu_goc`, mỗi nguồn một thư mục mang tên rõ ràng. Bảng nhãn `winter_classes.txt` nằm cùng nguồn Winter. Notebook tìm nguồn ở thư mục chung trước và lưu bảng đường dẫn cũ → mới trong báo cáo. Những tệp đã ở đúng vị trí được giữ nguyên khi chạy lại; các kết quả phân tích trước đây vẫn được giữ để đối chiếu.

**Hai chế độ chạy:**

- `xu_ly_du_lieu`: kiểm tra nguồn và lưu báo cáo. Bước 1 đọc thông tin nguồn; bước 2 xử lý ảnh YOLO1 theo quy mô đã chọn.
- `xem_bao_cao`: mở báo cáo đã lưu. Nếu chưa có kết quả, notebook thông báo phần còn thiếu và không tự chạy kiểm tra nguồn.

Khi cần tính mới, chọn chế độ `xu_ly_du_lieu`. Các lần đọc lại báo cáo có thể chuyển sang `xem_bao_cao`.


In [ ]:
# @title Kết nối Drive và chọn chế độ
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

CHE_DO = "xem_bao_cao" # @param ["xu_ly_du_lieu", "xem_bao_cao"]
THU_MUC_ML = "/content/drive/MyDrive/ML" # @param {type:"string"}
THU_MUC_DU_AN = "/content/drive/MyDrive/ML/Do_an_bien_so" # @param {type:"string"}

# Để trống để tự tìm nguồn, ưu tiên các thư mục nguồn chung của dự án.
# Nếu báo nhiều tệp có thể dùng hoặc không tìm thấy, nhập đường dẫn đầy đủ vào dòng tương ứng.
TEP_GLOBAL = "" # @param {type:"string"}
TEP_ROBOFLOW = "" # @param {type:"string"}
TEP_WINTER = "" # @param {type:"string"}
TEP_CCPD = "" # @param {type:"string"}
TEP_VNLP = "" # @param {type:"string"}

# Có thể để trống: notebook thử tên đã biết trong các thư mục thông dụng.
# Muốn so sánh đúng mục 7 hiện tại, bản notebook cũ cần được lưu trên Drive trước.
DUONG_DAN_NOTEBOOK_CU = "" # @param {type:"string"}

TEP_TUY_CHON = {"Global_v13": TEP_GLOBAL, "Roboflow": TEP_ROBOFLOW,
               "Winter2897": TEP_WINTER, "CCPD2019": TEP_CCPD, "VNLP": TEP_VNLP}
print("Chế độ:", "Kiểm tra nguồn và lưu báo cáo" if CHE_DO == "xu_ly_du_lieu" else "Xem báo cáo đã lưu")
print("Thư mục dự án:", THU_MUC_DU_AN)


### 1.2. Các tệp nguồn đã có đủ chưa?

Chúng ta kiểm tra vị trí, dung lượng và thông tin nhận diện của từng bộ trước khi phân tích ảnh. Global v13 và Roboflow v1 được xác nhận bằng hai thông tin đi cùng trong ZIP: đường dẫn dự án trong `README.dataset.txt` và số phiên bản trong `README.roboflow.txt`. Tên tệp hoặc số lượng ảnh riêng lẻ không đủ để xác nhận nguồn.

Đã tìm thấy đủ năm tệp nguồn và sắp xếp vào thư mục chung. Bảng dưới ghi vị trí hiện tại, còn bảng lịch sử sắp xếp lưu cả đường dẫn cũ và mới.

Bước này chỉ đọc thông tin tệp và danh sách trong ZIP để chạy nhanh. Ảnh, nhãn, trùng lặp và chất lượng sẽ được kiểm tra ở từng phần phân tích. Winter và CCPD còn cần kiểm tra nội dung chi tiết; CCPD chưa được quét toàn bộ ảnh hoặc giải nén ở bước này.


In [ ]:
# @title Các hàm kiểm tra nguồn — có thể thu gọn ô mã này
"""Bước 1: kiểm kê nguồn bằng thông tin tệp, chưa xử lý ảnh."""
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone, timedelta
from zipfile import ZipFile
import csv
import hashlib
import io
import json
import re
import time
import uuid
from urllib.parse import urlsplit

PHIEN_BAN = "buoc_1_1.2"
THU_MUC = {
    "nguon": "01_Du_lieu_goc", "train": "02_Du_lieu_huan_luyen",
    "bao_cao": "03_Bao_cao", "anh": "04_Anh_dai_dien",
    "notebook": "05_Notebook", "cu": "06_Ket_qua_cu",
}
NGUON = {
    "Global_v13": {"vai_tro": "YOLO1", "tep": "license-plate-dataset.zip",
        "du_an": "roboflow-universe-projects/license-plate-recognition-rxg4e", "ban": 13},
    "Roboflow": {"vai_tro": "YOLO2; OCR train/validation", "tep": "roboflow_ocr_v1_yolov8.zip",
        "du_an": "licence-plate-vn/ocr-stypd", "ban": 1},
    "Winter2897": {"vai_tro": "OCR train/validation", "tep": "yolo_plate_ocr_dataset.zip"},
    "CCPD2019": {"vai_tro": "OCR train/validation", "tep": "CCPD2019.tar.xz"},
    "VNLP": {"vai_tro": "OCR test riêng", "tep": "VNLP.zip"},
}
ANH = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SPLIT = {"train": "train", "valid": "val", "val": "val", "test": "test"}
BASELINE_7 = {
    "7.1": "dba6a81c5a1eb2b010af57931abc00a183f567670c18397c603291c829469868",
    "7.2": "bc89894fdc8a4d2b2b4c3e02c3f59983033b0ced40cfb51ac16af98331f1a29d",
    "7.3": "212676547a5a941ddd54587e49459f64cf05ee90038f93f848dd01a991bd4708",
    "7.4": "01e2bd476cfccd69d0b65bb420513d0767504c03f679b57182970266c49413ac"
}


DOI_CHIEU_BAN_DA_GUI = {'trang_thai': 'Đã đối chiếu bản notebook người dùng gửi ngày 20/11/2026', 'tep': 'Phan_tich_OCR32_ra_soat_va_lam_sach.ipynb (bản đính kèm đã đọc, không phải tệp vừa mở trên Drive)', 'sha256_notebook': '6fa6a13e24310105f52c8adbc39841b7cbedfa952e60350f7ccf087bb7f8276e', 'muc_7': [{'muc': '7.1', 'sha256': '2e54f938967a5189d14e75f06adbf7cfa76f3d05fc5de2a9551dd49e4d285256', 'so_voi_ban_luu_18_11': 'Khác hoặc chưa có mẫu đối chiếu'}, {'muc': '7.2', 'sha256': '21ccd794c21a5db56b215938dfe48014eefc115fc342a13631520fd0f3047d95', 'so_voi_ban_luu_18_11': 'Khác hoặc chưa có mẫu đối chiếu'}, {'muc': '7.3', 'sha256': '6181e606115667eea62e4ad9f784279ed77708261711d9d358803ebff5606152', 'so_voi_ban_luu_18_11': 'Khác hoặc chưa có mẫu đối chiếu'}, {'muc': '7.4', 'sha256': '01e2bd476cfccd69d0b65bb420513d0767504c03f679b57182970266c49413ac', 'so_voi_ban_luu_18_11': 'Giống'}], 'thieu_muc': [], 'ghi_chu': 'Mục 7.2 chỉ thay hàm run_incremental để gom nhiều biển trước khi lưu. Chưa xác minh ZIP dữ liệu thật, tốc độ trước/sau hay thay đổi sau lúc gửi tệp.'}


def _ma_cau_hinh(ml, du_an, tuy_chon, notebook_cu):
    data = {"phien_ban": PHIEN_BAN, "ml": str(Path(ml)), "du_an": str(Path(du_an)),
            "tuy_chon": dict(tuy_chon or {}), "notebook_cu": str(notebook_cu or "")}
    return hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()


def _ghi(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tam = path.with_name(path.name + "." + uuid.uuid4().hex + ".part")
    try:
        tam.write_text(text, encoding="utf-8")
        tam.replace(path)
    finally:
        tam.unlink(missing_ok=True)


def _doc_json_zip(path, basename):
    with ZipFile(path) as z:
        ds = [x for x in z.infolist() if PurePosixPath(x.filename).name == basename and not x.is_dir()]
        if len(ds) != 1:
            raise ValueError(f"Cần đúng một {basename}; tìm thấy {len(ds)}.")
        if ds[0].file_size > 1024 * 1024:
            raise ValueError(f"{basename} lớn bất thường; chưa đọc.")
        return json.loads(z.read(ds[0]).decode("utf-8-sig"))


def doc_cau_hinh_cu(ml):
    lan_cu = Path(ml) / "ket_qua_bien_so/20261113_155101_m8ns80u9"
    ds = [lan_cu / "bao_cao_phan_tich.zip", lan_cu / "ban_giao/bao_cao_phan_tich.zip"]
    for path in ds:
        if path.is_file():
            try:
                data = _doc_json_zip(path, "run_config.json")
                mapping = {"global": "Global_v13", "roboflow_ocr": "Roboflow", "winter": "Winter2897"}
                paths = {}
                for source in data.get("sources", []):
                    name = mapping.get(source.get("key"))
                    if name and source.get("file"):
                        paths[name] = source["file"]
                return {"tep_bao_cao": str(path), "duong_dan_nguon": paths,
                        "ghi_chu": "Đường dẫn từng được dùng, chưa xác nhận nội dung tệp hiện tại."}
            except Exception as e:
                return {"tep_bao_cao": str(path), "duong_dan_nguon": {}, "loi": str(e)}
    return {"tep_bao_cao": "", "duong_dan_nguon": {}, "ghi_chu": "Chưa tìm được báo cáo cũ."}


def tim_tep(ten, ml, du_an, tuy_chon, cau_hinh_cu):
    ghi_de = str((tuy_chon or {}).get(ten, "")).strip()
    if ghi_de:
        p = Path(ghi_de).expanduser()
        if not p.is_absolute():
            return None, [], "Đường dẫn nhập tay cần là đường dẫn đầy đủ."
        return (p, [str(p)], "") if p.is_file() else (None, [str(p)], "Không thấy tệp đã chỉ định.")
    spec = NGUON[ten]
    folder = Path(du_an) / THU_MUC["nguon"] / ten
    ds = [folder / spec["tep"]]
    if folder.is_dir():
        ds.extend(sorted(folder.glob("*.tar.xz" if ten == "CCPD2019" else "*.zip")))
    if ten in cau_hinh_cu.get("duong_dan_nguon", {}):
        ds.append(Path(cau_hinh_cu["duong_dan_nguon"][ten]))
    if ten in {"CCPD2019", "VNLP"}:
        ds.append(Path(ml) / "ocr32_bo_sung/sources" / spec["tep"])
    else:
        ds.append(Path(ml) / spec["tep"])
    if ten == "Global_v13":
        ds.append(Path(ml) / "License Plate Recognition.v13.yolov8.zip")
    co = {}
    for p in ds:
        if p.is_file():
            co[str(p.resolve())] = p
    if len(co) == 1:
        return next(iter(co.values())), list(co), ""
    if len(co) > 1:
        return None, list(co), "Có nhiều tệp có thể dùng; cần chỉ định đường dẫn, chưa tự chọn."
    return None, [], "Chưa tìm thấy tệp tại các vị trí đã khai báo."


def _split(parts):
    found = set()
    for i in range(len(parts) - 1):
        if parts[i].lower() in SPLIT and parts[i + 1].lower() in {"images", "labels"}:
            found.add(SPLIT[parts[i].lower()])
        if parts[i].lower() in {"images", "labels"} and parts[i + 1].lower() in SPLIT:
            found.add(SPLIT[parts[i + 1].lower()])
    return next(iter(found)) if len(found) == 1 else "chua_xac_dinh"


def kiem_tra_zip(path, ten):
    counts, labels = Counter(), Counter()
    readmes, urls, yaml_names = [], set(), []
    bad_paths = duplicates = 0
    names = set()
    with ZipFile(path) as z:
        for info in z.infolist():
            if info.is_dir():
                continue
            p = PurePosixPath(info.filename)
            if p.is_absolute() or ".." in p.parts or "\\" in info.filename:
                bad_paths += 1
                continue
            if info.filename in names:
                duplicates += 1
            names.add(info.filename)
            if any(x.startswith(".") or x == "__MACOSX" for x in p.parts):
                continue
            split = _split(p.parts[:-1])
            ext = p.suffix.lower()
            if ext in ANH:
                counts[split] += 1
            elif ext == ".txt" and any(x.lower() == "labels" for x in p.parts[:-1]):
                labels[split] += 1
            if p.name.lower() in {"data.yaml", "data.yml"}:
                yaml_names.append(info.filename)
            if p.name.lower().startswith("readme") and len(readmes) < 12 and info.file_size <= 128 * 1024:
                text = z.read(info).decode("utf-8-sig", errors="replace")
                readmes.append({"tep": info.filename, "noi_dung": text})
                urls.update(re.findall(r"https?://universe\.roboflow\.com/[^\s<>\"']+", text))
    identity = "Chưa xác nhận bằng nội dung"
    evidence = ""
    spec = NGUON[ten]
    if "du_an" in spec:
        project_urls, url_versions, header_versions = [], set(), set()
        for url in sorted(urls):
            parsed = urlsplit(url.rstrip(".,)]}"))
            if parsed.hostname != "universe.roboflow.com":
                continue
            path_part = parsed.path.strip("/")
            if path_part == spec["du_an"]:
                project_urls.append(url)
            match = re.fullmatch(re.escape(spec["du_an"]) + r"/(?:dataset/)?([0-9]+)", path_part)
            if match:
                url_versions.add(int(match.group(1)))
                if int(match.group(1)) == spec["ban"]:
                    evidence = url
        for item in readmes:
            if PurePosixPath(item["tep"]).name.lower() == "readme.roboflow.txt":
                first_line = next((line.strip() for line in item["noi_dung"].splitlines() if line.strip()), "")
                match = re.fullmatch(r".+?\s+-\s+v([0-9]+)(?:\s+.*)?", first_line)
                if match:
                    header_versions.add(int(match.group(1)))
        conflict = any(v != spec["ban"] for v in url_versions | header_versions)
        if not conflict and (evidence or (project_urls and header_versions == {spec["ban"]})):
            identity = "Khớp nguồn và phiên bản trong README"
            if not evidence:
                evidence = project_urls[0] + f" ; README.roboflow.txt ghi v{spec['ban']}"
        elif urls:
            identity = "README chưa khớp nguồn và phiên bản yêu cầu"
    elif ten == "VNLP" and "detection/list_one_row.txt" in names:
        identity = "Khớp cấu trúc VNLP; chưa kiểm tra toàn vẹn"
        evidence = "detection/list_one_row.txt"
    return {"xac_nhan": identity, "bang_chung": evidence, "so_anh": dict(counts),
            "so_nhan": dict(labels), "tep_yaml": yaml_names, "readme": readmes,
            "lien_ket_nguon": sorted(urls), "duong_dan_bat_thuong": bad_paths,
            "ten_tep_lap": duplicates}


def _chuan_hoa_cell(source):
    return "\n".join(x.rstrip() for x in source.strip().splitlines())


def doi_chieu_notebook(ml, du_an, chi_dinh=""):
    name = "Phan_tich_OCR32_ra_soat_va_lam_sach.ipynb"
    if chi_dinh:
        ds = [Path(chi_dinh)]
    else:
        ds = [Path(ml).parent / "Colab Notebooks" / name, Path(ml) / name,
              Path(du_an) / THU_MUC["notebook"] / name]
    co = list({str(p.resolve()): p for p in ds if p.is_file()}.values())
    if not co and not chi_dinh and DOI_CHIEU_BAN_DA_GUI:
        return json.loads(json.dumps(DOI_CHIEU_BAN_DA_GUI, ensure_ascii=False))
    if len(co) != 1:
        return {"trang_thai": "Chưa đối chiếu", "ly_do": "Chưa tìm đúng một bản notebook cũ; có thể nhập đường dẫn ở ô cấu hình.",
                "tep_tim_thay": [str(p) for p in co]}
    p = co[0]
    if p.stat().st_size > 64 * 1024 * 1024:
        return {"trang_thai": "Chưa đối chiếu", "ly_do": "Notebook lớn hơn 64 MiB; cần đọc bản đã xóa output để kiểm tra nhẹ."}
    try:
        raw = p.read_bytes()
        nb = json.loads(raw)
        rows = []
        for c in nb.get("cells", []):
            if c.get("cell_type") != "code":
                continue
            src = c.get("source", "")
            src = "".join(src) if isinstance(src, list) else src
            title = re.search(r"^\s*#\s*@title\s+(7\.[1-4])\b", src, re.M)
            if title:
                key = title.group(1)
                digest = hashlib.sha256(_chuan_hoa_cell(src).encode()).hexdigest()
                baseline = BASELINE_7.get(key)
                rows.append({"muc": key, "sha256": digest,
                             "so_voi_ban_luu_18_11": "Giống" if digest == baseline else "Khác hoặc chưa có mẫu đối chiếu"})
        matched = {r["muc"] for r in rows}
        return {"trang_thai": "Đã đọc bản notebook lưu trên Drive", "tep": str(p),
                "sha256_notebook": hashlib.sha256(raw).hexdigest(), "muc_7": rows,
                "thieu_muc": sorted(set(BASELINE_7) - matched),
                "ghi_chu": "So sánh mã chỉ xác định có thay đổi; chưa chứng minh tối ưu đúng hay nhanh. Thay đổi chưa lưu trên Colab không nằm trong tệp này."}
    except Exception as e:
        return {"trang_thai": "Chưa đối chiếu", "tep": str(p), "ly_do": str(e)}


def tao_thu_muc(du_an):
    du_an = Path(du_an)
    for folder in THU_MUC.values():
        (du_an / folder).mkdir(parents=True, exist_ok=True)
    for name in NGUON:
        (du_an / THU_MUC["nguon"] / name).mkdir(exist_ok=True)
    for name in ("YOLO1", "YOLO2", "OCR"):
        (du_an / THU_MUC["train"] / name).mkdir(exist_ok=True)


def ket_luan(data):
    rows = data["nguon"]
    co = sum(bool(r["duong_dan"]) and not r.get("loi") for r in rows)
    global_row = next(r for r in rows if r["nguon"] == "Global_v13")
    detail = data["chi_tiet"].get("Global_v13", {})
    if detail.get("xac_nhan") == "Khớp nguồn và phiên bản trong README":
        global_text = "Global v13 đã khớp đường dẫn dự án và phiên bản trong README."
    elif global_row.get("duong_dan"):
        global_text = "Đã tìm thấy tệp có thể là Global v13, nhưng chưa đủ bằng chứng nội dung để xác nhận."
    else:
        global_text = "Chưa xác định được tệp Global v13; cần xem các đường dẫn gợi ý hoặc khai báo đường dẫn."
    return (f"Đã tìm được và đọc thông tin sơ bộ của {co}/{len(rows)} nguồn. {global_text}\n\n"
            "Số ảnh ở bước này được đếm từ danh sách tệp trong ZIP, chưa kiểm tra ảnh/nhãn hợp lệ, "
            "chưa loại trùng và chưa phải số mẫu huấn luyện cuối cùng. CCPD chưa được quét ảnh hoặc giải nén.\n\n"
            "Báo cáo nằm trong thư mục chung. Vị trí nguồn hiện tại được ghi trong bảng; "
            "lịch sử sắp xếp được lưu riêng nếu đã thực hiện.\n\n"
            "Bước tiếp theo là triển khai thống kê YOLO1 sau khi nguồn Global đã được xác nhận. "
            "Các nguồn OCR còn chờ kiểm tra nội dung sẽ được kiểm tra chi tiết ở bước tương ứng. "
            "Bộ dữ liệu huấn luyện mới và quy tắc lọc OCR mới chưa được tạo ở bước 1.")


def thuc_hien_buoc_1(ml, du_an, che_do="xu_ly_du_lieu", tuy_chon=None, notebook_cu=""):
    if che_do not in {"xu_ly_du_lieu", "xem_bao_cao"}:
        raise ValueError("Chế độ không hợp lệ.")
    ml, du_an = Path(ml), Path(du_an)
    folder = du_an / THU_MUC["bao_cao"] / "01_Kiem_tra_nguon"
    saved = folder / "ket_qua_kiem_tra.json"
    key = _ma_cau_hinh(ml, du_an, tuy_chon, notebook_cu)
    if che_do == "xem_bao_cao":
        if not saved.is_file():
            return {"trang_thai": "Chưa có kết quả", "thong_bao": "Chọn xu_ly_du_lieu và chạy bước 1 một lần. Chế độ xem không tự xử lý nguồn."}
        data = json.loads(saved.read_text(encoding="utf-8"))
        if data.get("phien_ban") != PHIEN_BAN or data.get("ma_cau_hinh") != key:
            return {"trang_thai": "Cấu hình đã thay đổi", "thong_bao": "Kết quả đã lưu thuộc cấu hình khác; chọn xu_ly_du_lieu để cập nhật."}
        data["cach_doc"] = "Đọc báo cáo đã lưu; chưa kiểm tra lại sự thay đổi của các tệp nguồn."
        return data
    if not ml.is_dir():
        raise FileNotFoundError(f"Chưa thấy thư mục nguồn: {ml}. Kiểm tra kết nối Drive.")
    started = time.perf_counter()
    tao_thu_muc(du_an)
    old = doc_cau_hinh_cu(ml)
    rows, details, splits = [], {}, []
    used = {}
    for name, spec in NGUON.items():
        print(f"Đang kiểm tra {name}…", flush=True)
        t0 = time.perf_counter()
        path, candidates, error = tim_tep(name, ml, du_an, tuy_chon, old)
        row = {"nguon": name, "vai_tro": spec["vai_tro"], "duong_dan": str(path) if path else "",
               "dung_luong_GiB": None, "tinh_trang": error, "loi": error,
               "tep_co_the_dung": candidates}
        if path:
            try:
                resolved = str(path.resolve())
                if resolved in used:
                    raise ValueError(f"Cùng tệp đã được khai báo cho nguồn {used[resolved]}; chưa dùng cho {name}.")
                used[resolved] = name
                row["dung_luong_GiB"] = round(path.stat().st_size / 2**30, 3)
                if name == "CCPD2019":
                    with path.open("rb") as f:
                        if f.read(6) != b"\xfd7zXZ\x00":
                            raise ValueError("Tệp không có phần đầu định dạng XZ như dự kiến.")
                    detail = {"xac_nhan": "Có tệp XZ; chưa xác nhận nội dung CCPD2019",
                              "so_anh": {}, "so_nhan": {}, "ghi_chu": "Không quét TAR.XZ ở bước kiểm tra nhẹ."}
                else:
                    detail = kiem_tra_zip(path, name)
                details[name] = detail
                row["tinh_trang"] = detail["xac_nhan"]
                for split, count in detail["so_anh"].items():
                    splits.append({"nguon": name, "tap": split, "so_tep_anh": count,
                                   "so_tep_nhan": detail["so_nhan"].get(split, 0)})
            except Exception as e:
                row["loi"] = str(e)
                row["tinh_trang"] = "Cần kiểm tra: " + str(e)
        row["thoi_gian_giay"] = round(time.perf_counter() - t0, 3)
        rows.append(row)
    notebook = doi_chieu_notebook(ml, du_an, notebook_cu)
    sidecar = du_an / THU_MUC["nguon"] / "Winter2897/winter_classes.txt"
    if not sidecar.is_file():
        sidecar = Path(ml) / "winter_classes.txt"
    data = {"phien_ban": PHIEN_BAN, "ma_cau_hinh": key,
            "thoi_diem": datetime.now(timezone(timedelta(hours=7))).isoformat(timespec="seconds"),
            "trang_thai": "Đã kiểm tra sơ bộ", "nguon": rows, "chia_tap_goc": splits,
            "chi_tiet": details, "cau_hinh_cu": old, "notebook_cu": notebook,
            "bang_nhan_winter_cu": str(sidecar) if sidecar.is_file() else "Chưa tìm thấy",
            "thoi_gian_tong_giay": round(time.perf_counter() - started, 3)}
    data["ket_luan"] = ket_luan(data)
    for filename, records in [("danh_sach_nguon.csv", rows), ("chia_tap_goc.csv", splits)]:
        fields = list(records[0]) if records else ["nguon", "tap", "so_tep_anh", "so_tep_nhan"]
        buf = io.StringIO(newline="")
        writer = csv.DictWriter(buf, fieldnames=fields)
        writer.writeheader()
        for record in records:
            writer.writerow({k: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v for k, v in record.items()})
        _ghi(folder / filename, "\ufeff" + buf.getvalue())
    _ghi(folder / "ket_luan.md", "# Kết luận kiểm tra nguồn\n\n" + data["ket_luan"] + "\n")
    _ghi(folder / "doi_chieu_notebook_cu.json", json.dumps(notebook, ensure_ascii=False, indent=2))
    _ghi(saved, json.dumps(data, ensure_ascii=False, indent=2))
    return data


def sap_xep_nguon(ml, du_an, ket_qua):
    """Đổi vị trí trong cùng Drive bằng rename; không sao chép hay giải nén tệp lớn."""
    ml, du_an = Path(ml), Path(du_an)
    rows = {r["nguon"]: r for r in ket_qua.get("nguon", [])}
    if set(rows) != set(NGUON):
        raise ValueError("Cần kiểm tra đủ năm nguồn trước khi sắp xếp.")
    for name in ("Global_v13", "Roboflow"):
        if rows[name].get("tinh_trang") != "Khớp nguồn và phiên bản trong README":
            raise ValueError("Chưa xác nhận đúng nguồn/phiên bản: " + name)
    plan = []
    for name, spec in NGUON.items():
        row = rows[name]
        if row.get("loi") or not row.get("duong_dan"):
            raise ValueError("Nguồn đang thiếu hoặc có lỗi: " + name)
        src = Path(row["duong_dan"])
        dst = du_an / THU_MUC["nguon"] / name / spec["tep"]
        plan.append({"nguon": name, "cu": str(src), "moi": str(dst)})
    class_file = ml / "winter_classes.txt"
    new_class = du_an / THU_MUC["nguon"] / "Winter2897/winter_classes.txt"
    if class_file.is_file():
        plan.append({"nguon": "Nhãn Winter", "cu": str(class_file), "moi": str(new_class)})
    # Kiểm tra toàn bộ đích trước khi chuyển bất kỳ tệp nào; không ghi đè.
    for item in plan:
        src, dst = Path(item["cu"]), Path(item["moi"])
        if not src.is_file():
            raise FileNotFoundError(src)
        same = src.resolve() == dst.resolve()
        if not same and dst.exists():
            raise FileExistsError("Đích đã có tệp; chưa di chuyển: " + str(dst))
        if not src.resolve().is_relative_to(ml.resolve()) or not dst.resolve().is_relative_to(du_an.resolve()):
            raise ValueError("Đường dẫn nằm ngoài thư mục dự án đã khai báo.")
        item["dung_luong_byte"] = src.stat().st_size
        item["trang_thai"] = "Đã ở thư mục chung" if same else "Chờ chuyển"
    folder = du_an / THU_MUC["bao_cao"] / "01_Kiem_tra_nguon"
    log_path = folder / "lich_su_sap_xep_nguon.json"
    ledger = json.loads(log_path.read_text(encoding="utf-8")) if log_path.is_file() else []
    receipt = {"thoi_diem": datetime.now(timezone(timedelta(hours=7))).isoformat(timespec="seconds"),
               "thao_tac": "rename trong cùng Drive; không sao chép ảnh", "tep": plan}
    ledger.append(receipt)
    _ghi(log_path, json.dumps(ledger, ensure_ascii=False, indent=2))
    for item in plan:
        if item["trang_thai"] == "Đã ở thư mục chung":
            continue
        src, dst = Path(item["cu"]), Path(item["moi"])
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            src.rename(dst)
            if not dst.is_file() or dst.stat().st_size != item["dung_luong_byte"] or src.exists():
                raise OSError("Cần kiểm tra kết quả di chuyển: " + str(dst))
            item["trang_thai"] = "Đã chuyển, dung lượng giữ nguyên"
        except OSError as e:
            item["trang_thai"] = "Cần kiểm tra: " + str(e)
            _ghi(log_path, json.dumps(ledger, ensure_ascii=False, indent=2))
            raise
        _ghi(log_path, json.dumps(ledger, ensure_ascii=False, indent=2))
    return receipt


In [ ]:
# @title Kiểm tra nguồn hoặc mở kết quả đã lưu
from IPython.display import display, Markdown
import pandas as pd

KET_QUA = thuc_hien_buoc_1(
    THU_MUC_ML, THU_MUC_DU_AN, CHE_DO, TEP_TUY_CHON, DUONG_DAN_NOTEBOOK_CU
)
if "nguon" not in KET_QUA:
    print(KET_QUA["thong_bao"])
else:
    print("Thời điểm kiểm tra:", KET_QUA["thoi_diem"])
    if KET_QUA.get("cach_doc"):
        print(KET_QUA["cach_doc"])
    bang = pd.DataFrame(KET_QUA["nguon"])
    with pd.option_context("display.max_colwidth", None):
        display(bang[["nguon", "vai_tro", "dung_luong_GiB", "tinh_trang", "duong_dan"]].rename(columns={
            "nguon": "Nguồn", "vai_tro": "Sử dụng cho", "dung_luong_GiB": "Dung lượng (GiB)",
            "tinh_trang": "Kết quả kiểm tra", "duong_dan": "Vị trí tệp hiện tại"}))
    for row in KET_QUA["nguon"]:
        if not row["duong_dan"] and row["tep_co_the_dung"]:
            print("Các tệp cần chọn cho", row["nguon"] + ":")
            for path in row["tep_co_the_dung"]:
                print(" •", path)
    print("Thời gian kiểm tra lần đã lưu:", KET_QUA["thoi_gian_tong_giay"], "giây")


In [ ]:
# @title Sắp xếp các nguồn vào thư mục chung
SAP_XEP_NGUON = False # @param {type:"boolean"}

def xem_vi_tri_nguon(bien_nhan):
    bang_vi_tri = []
    for r in bien_nhan["tep"]:
        bang_vi_tri.append({"Nguồn": r["nguon"],
                           "Vị trí dưới thư mục dự án": str(Path(r["moi"]).relative_to(THU_MUC_DU_AN)),
                           "Kết quả": r["trang_thai"]})
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(bang_vi_tri))

if CHE_DO == "xu_ly_du_lieu" and SAP_XEP_NGUON:
    bien_nhan = sap_xep_nguon(THU_MUC_ML, THU_MUC_DU_AN, KET_QUA)
    xem_vi_tri_nguon(bien_nhan)
    KET_QUA = thuc_hien_buoc_1(THU_MUC_ML, THU_MUC_DU_AN, CHE_DO,
                              TEP_TUY_CHON, DUONG_DAN_NOTEBOOK_CU)
    print("Đã kiểm tra lại các nguồn sau khi sắp xếp. Báo cáo đã cập nhật đường dẫn mới.")
    print("Những tệp đã ở đúng vị trí được giữ nguyên khi chạy lại.")
else:
    print("Đang xem kết quả hoặc chưa bật sắp xếp; không di chuyển tệp.")
    duong_dan_lich_su = Path(THU_MUC_DU_AN) / "03_Bao_cao/01_Kiem_tra_nguon/lich_su_sap_xep_nguon.json"
    if duong_dan_lich_su.is_file():
        lan_sap_xep = json.loads(duong_dan_lich_su.read_text(encoding="utf-8"))[-1]
        print("Lần sắp xếp đã lưu:", lan_sap_xep["thoi_diem"])
        xem_vi_tri_nguon(lan_sap_xep)


### 1.3. Các bộ YOLO đang được chia như thế nào?

Đối với YOLO1 và YOLO2, chúng ta ưu tiên giữ cách chia `train`, `valid/val`, `test` có sẵn trong bộ tải xuống. Bảng dưới đếm số tệp ảnh và nhãn trong từng tập từ danh sách ZIP.

**Cách đọc:** đây là quy mô ban đầu, chưa phải số mẫu hợp lệ sau làm sạch. Bảng không khẳng định mỗi ảnh đã có đúng nhãn tương ứng. Tên `valid` được trình bày thống nhất thành `val`; vị trí tệp nguồn vẫn giữ nguyên.


In [ ]:
# @title Xem số tệp ảnh và nhãn trong các tập gốc
if "nguon" in KET_QUA:
    rows = [r for r in KET_QUA["chia_tap_goc"] if r["nguon"] in {"Global_v13", "Roboflow"}]
    if rows:
        display(pd.DataFrame(rows).rename(columns={
            "nguon": "Nguồn", "tap": "Tập", "so_tep_anh": "Số tệp ảnh", "so_tep_nhan": "Số tệp nhãn"}))
    else:
        print("Chưa có số đếm YOLO từ tệp nguồn. Xem tình trạng nguồn ở bảng trên; không diễn giải là dataset có 0 ảnh.")


### 1.4. Mục 7 của notebook cũ đã thay đổi những gì?

Đã đọc bản notebook người dùng gửi ngày 20/11 và đối chiếu với bản tham chiếu ngày 18/11: mục 7.1, 7.2 và 7.3 có thay đổi; mục 7.4 giữ nguyên. Phần thay đổi trong mã xử lý 7.2 tập trung vào việc gom nhiều biển hơn trước mỗi lần lưu tiến độ. Các hàm cắt ký tự, lọc CCPD và kiểm tra trùng pixel không thay đổi.

Log đã lưu ghi lượt bổ sung tiếp xuất ZIP thành công: train **334.826**, validation **19.928**, test **19.306**; còn thiếu **23.062** ảnh so với mục tiêu theo lớp. Đây là kết quả của phương án cũ, có VNLP trong train; chưa phải dữ liệu của phương án mới và chưa được kiểm tra lại nội dung ZIP trên Drive.

Nếu tìm thấy bản `.ipynb` cũ trên Drive, bước này đọc phần mã 7.1–7.4 và so sánh, không chạy các ô đó. Nếu chưa tìm thấy và không nhập đường dẫn, notebook hiển thị kết quả đối chiếu bản đính kèm đã đọc, kèm nhãn phân biệt rõ.

Kết quả **giống/khác** chỉ cho biết mã có thay đổi hay không. Nó chưa chứng minh quy tắc lọc đúng, ảnh cắt khớp nhãn hoặc tốc độ đã cải thiện. Muốn đối chiếu một bản mới hơn bản đã gửi, nhập đường dẫn đầy đủ trong ô cấu hình rồi chạy kiểm tra lại. Những thay đổi chưa lưu trên Colab chưa có trong bản tệp trên Drive.


In [ ]:
# @title Xem tình trạng đối chiếu notebook cũ
if "notebook_cu" in KET_QUA:
    doi_chieu = KET_QUA["notebook_cu"]
    print(doi_chieu["trang_thai"])
    if doi_chieu.get("tep"):
        print("Tệp đã đọc:", doi_chieu["tep"])
    if doi_chieu.get("ly_do"):
        print(doi_chieu["ly_do"])
    if doi_chieu.get("muc_7"):
        display(pd.DataFrame(doi_chieu["muc_7"])[["muc", "so_voi_ban_luu_18_11"]].rename(columns={
            "muc": "Mục", "so_voi_ban_luu_18_11": "So với bản lưu ngày 18/11"}))
    if doi_chieu.get("thieu_muc"):
        print("Chưa nhận diện được các mục:", ", ".join(doi_chieu["thieu_muc"]))


### 1.5. Kết luận sau khi kiểm tra nguồn

Kết luận dưới đây được tạo từ kết quả kiểm tra thực tế, để phân biệt nguồn đã tìm thấy, nguồn đã có bằng chứng nhận diện và phần còn phải kiểm tra.


In [ ]:
# @title Đọc kết luận và vị trí báo cáo
if "ket_luan" in KET_QUA:
    display(Markdown(KET_QUA["ket_luan"]))
    print("Báo cáo:", Path(THU_MUC_DU_AN) / "03_Bao_cao/01_Kiem_tra_nguon")


## 2. Phân tích dữ liệu phát hiện biển số — YOLO1

YOLO1 học cách tìm vùng biển số trên ảnh. Phần này chỉ dùng **Global v13**, với ba tập có sẵn trong bộ tải xuống. Chúng ta tìm hiểu số lượng, kích thước ảnh và biển số, kiểm tra lỗi và trùng lặp, rồi trích ảnh mẫu để đối chiếu.

**Có thể chạy thử 300 ảnh trước khi xét toàn bộ nguồn.** Số liệu mẫu thử và toàn nguồn được lưu ở hai nơi riêng; khi đặt số ảnh thử bằng `0`, chương trình mới xét toàn bộ nguồn. Chưa tự loại ảnh hoặc thay đổi cách chia tập ở lượt phân tích này. Việc đó sẽ dựa trên các nhóm lỗi/trùng được xác nhận sau khi xem ảnh.

### 2.1. Dữ liệu có bao nhiêu ảnh và biển số?

Bảng đầu tiên phân biệt số ảnh có trong nguồn với số ảnh đã được kiểm tra. Một ảnh có thể chứa nhiều biển, vì vậy số ảnh và số khung không được dùng thay thế cho nhau. Mã hỗ trợ dài bên dưới có thể thu gọn khi đọc báo cáo.


In [ ]:
# @title Các hàm phân tích YOLO1 — có thể thu gọn
"""Phân tích Global v13; chỉ đọc nguồn, chưa quyết định loại mẫu/đổi tập."""
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from zipfile import ZipFile
from datetime import datetime, timezone, timedelta
from urllib.parse import urlsplit
from concurrent.futures import ThreadPoolExecutor
import os
import csv, gzip, hashlib, io, json, math, re, shutil, time, uuid
import numpy as np
import PIL
from PIL import Image, features

Y1_VERSION = 'yolo1_2.0.0'
Y1_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
Y1_SPLIT = {'train': 'train', 'val': 'val', 'valid': 'val', 'test': 'test'}


def y1_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.part')
    try:
        tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)


def y1_csv(path, rows, fields=None):
    rows = list(rows)
    if not fields:
        fields = list(rows[0]) if rows else ['ghi_chu']
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
    path = Path(path)
    temp = path.with_name(path.name + '.part')
    temp.write_text(buf.getvalue(), encoding='utf-8-sig')
    temp.replace(path)


def y1_decode_versions():
    return {'Pillow': PIL.__version__, 'numpy': np.__version__,
            'JPEG': getattr(Image.core, 'jpeglib_version', None),
            'libjpeg_turbo': features.version('libjpeg_turbo')}


def y1_index(z):
    infos = [i for i in z.infolist() if not i.is_dir()]
    names = [i.filename for i in infos]
    if len(names) != len(set(names)):
        raise ValueError('ZIP có tên tệp lặp; cần xem lại trước khi ghép ảnh–nhãn.')
    images, labels, other = [], {}, []
    for info in infos:
        p = PurePosixPath(info.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in info.filename:
            raise ValueError('Đường dẫn trong ZIP không an toàn: ' + info.filename)
        parts = p.parts
        found = [(j, Y1_SPLIT[parts[j-1]]) for j in range(1, len(parts)-1)
                 if parts[j] in {'images', 'labels'} and parts[j-1] in Y1_SPLIT]
        if len(found) != 1:
            if p.suffix.lower() in Y1_EXT:
                other.append(info.filename)
            continue
        j, split = found[0]
        key = ('/'.join(parts[:j-1]), split, '/'.join(parts[j+1:]), p.suffix.lower())
        rel_stem = str(PurePosixPath(key[2]).with_suffix(''))
        pair_key = (key[0], split, rel_stem)
        if parts[j] == 'images' and p.suffix.lower() in Y1_EXT:
            images.append({'anh': info.filename, 'tap': split, 'key': pair_key,
                           'kich_thuoc_tep': info.file_size, 'crc': info.CRC})
        elif parts[j] == 'labels' and p.suffix.lower() == '.txt':
            if pair_key in labels:
                raise ValueError('Có nhiều tệp nhãn cho cùng ảnh: ' + info.filename)
            labels[pair_key] = info.filename
    keys = [x['key'] for x in images]
    if len(keys) != len(set(keys)):
        raise ValueError('Có nhiều ảnh cùng tên gốc và tập; chưa thể ghép nhãn chắc chắn.')
    if other:
        raise ValueError('Có ảnh ngoài cấu trúc train/val/test images: ' + repr(other[:3]))
    for x in images:
        x['nhan'] = labels.get(x.pop('key'), '')
    used = {x['nhan'] for x in images}
    orphan = sorted(set(labels.values()) - used)
    images.sort(key=lambda x: (x['tap'], x['anh']))
    # Ràng buộc tiến độ với metadata mọi thành viên; đây không phải SHA nội dung toàn ZIP.
    digest = hashlib.sha256()
    for i in sorted(infos, key=lambda x: x.filename):
        digest.update(json.dumps([i.filename, i.file_size, i.compress_size, i.CRC],
                                 ensure_ascii=False).encode())
    return images, orphan, digest.hexdigest()


def y1_source_identity(z):
    project = 'roboflow-universe-projects/license-plate-recognition-rxg4e'
    root_found, versions, exact = False, set(), False
    for info in z.infolist():
        name = PurePosixPath(info.filename).name.lower()
        if not name.startswith('readme') or info.file_size > 128*1024:
            continue
        text = z.read(info).decode('utf-8-sig', errors='replace')
        for url in re.findall(r'https?://universe\.roboflow\.com/[^\s<>"\']+', text):
            parsed = urlsplit(url.rstrip('.,)]}'))
            path = parsed.path.strip('/')
            if parsed.hostname != 'universe.roboflow.com':
                continue
            root_found |= path == project
            match = re.fullmatch(re.escape(project) + r'/(?:dataset/)?([0-9]+)', path)
            if match:
                versions.add(int(match.group(1)))
                exact |= int(match.group(1)) == 13
        if name == 'readme.roboflow.txt':
            first = next((x.strip() for x in text.splitlines() if x.strip()), '')
            match = re.fullmatch(r'.+?\s+-\s+v([0-9]+)(?:\s+.*)?', first)
            if match:
                versions.add(int(match.group(1)))
    if not (versions == {13} and (root_found or exact)):
        raise ValueError('README chưa xác nhận đúng Global v13 hoặc có bằng chứng phiên bản mâu thuẫn.')


def y1_parse(text, n_classes):
    boxes, issues, seen, duplicates = [], [], set(), 0
    for line_no, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        try:
            if len(parts) != 5:
                raise ValueError('can_5_cot')
            vals = list(map(float, parts))
            if not all(math.isfinite(v) for v in vals):
                raise ValueError('so_khong_huu_han')
            cls, x, y, w, h = vals
            if cls != int(cls) or not 0 <= cls < n_classes:
                raise ValueError('lop_khong_hop_le')
            if w <= 0 or h <= 0 or min(x-w/2, y-h/2) < -1e-6 or max(x+w/2, y+h/2) > 1+1e-6:
                raise ValueError('khung_ngoai_anh_hoac_khong_co_dien_tich')
            box = [int(cls), x, y, w, h]
            key = tuple(box)
            duplicates += int(key in seen)
            seen.add(key)
            boxes.append(box)
        except ValueError as e:
            issues.append(f'dong_{line_no}:{e}')
    return boxes, issues, duplicates


def y1_quality(image):
    image = image.convert('L')
    image.thumbnail((256, 256))
    a = np.asarray(image, dtype=np.float32)
    lap = (a[:-2, 1:-1] + a[2:, 1:-1] + a[1:-1, :-2] + a[1:-1, 2:]
           - 4*a[1:-1, 1:-1]) if min(a.shape) >= 3 else np.array([0.])
    return {'sang': round(float(a.mean()), 4), 'tuong_phan': round(float(a.std()), 4),
            'net': round(float(lap.var()), 4)}


def y1_one(z, item, n_classes):
    row = dict(item)
    row.update(loi_anh='', loi_nhan=[], so_khung=0, khung_lap=0, khung=[],
               rong=None, cao=None, pixel_sha256='', nhom_ten='', dhash='')
    p = PurePosixPath(item['anh'])
    if re.search(r'\.rf\.[0-9a-fA-F]{16,}$', p.stem):
        row['nhom_ten'] = p.stem.rsplit('.rf.', 1)[0]
    try:
        if not item['nhan']:
            row['loi_nhan'] = ['thieu_tep_nhan']
        elif z.getinfo(item['nhan']).file_size > 1024*1024:
            row['loi_nhan'] = ['tep_nhan_qua_lon']
        else:
            row['khung'], row['loi_nhan'], row['khung_lap'] = y1_parse(
                z.read(item['nhan']).decode('utf-8-sig'), n_classes)
        row['so_khung'] = len(row['khung'])
    except (UnicodeError, OSError, ValueError) as e:
        row['loi_nhan'] = ['khong_doc_duoc_nhan:' + type(e).__name__]
    try:
        raw = z.read(item['anh'])  # CRC ZIP được kiểm tra cho mỗi thành viên đã đọc.
        with Image.open(io.BytesIO(raw)) as im:
            im.load()
            row['exif_huong'] = int(im.getexif().get(274, 1))
            rgb = im.convert('RGB')  # Không tự xoay ảnh: nhãn YOLO dùng khung pixel lưu trong tệp.
        w, h = rgb.size
        row['rong'], row['cao'] = w, h
        row['pixel_sha256'] = hashlib.sha256(f'RGB:{w}:{h}:'.encode() + rgb.tobytes()).hexdigest()
        small = np.asarray(rgb.convert('L').resize((9, 8)), dtype=np.int16)
        row['dhash'] = np.packbits(small[:, 1:] > small[:, :-1]).tobytes().hex()
        row['chat_luong_anh'] = y1_quality(rgb)
        if row['khung']:
            b = max(row['khung'], key=lambda b: b[3]*b[4])
            x, y, bw, bh = b[1:]
            rect = (max(0, int((x-bw/2)*w)), max(0, int((y-bh/2)*h)),
                    min(w, math.ceil((x+bw/2)*w)), min(h, math.ceil((y+bh/2)*h)))
            row['chat_luong_bien_lon_nhat'] = y1_quality(rgb.crop(rect))
    except Exception as e:
        # Một ảnh lỗi không làm mất cả phần tiến độ. Không bắt KeyboardInterrupt.
        row['loi_anh'] = type(e).__name__ + ': ' + str(e)[:200]
    return row


def y1_groups(rows):
    parent = list(range(len(rows)))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(a, b):
        a, b = root(a), root(b)
        if a != b:
            parent[b] = a
    by = {k: defaultdict(list) for k in ['pixel_sha256', 'nhom_ten', 'dhash']}
    for i, r in enumerate(rows):
        for key in by:
            if r.get(key):
                by[key][r[key]].append(i)
    details, summary = [], []
    for key, groups in by.items():
        dup = [ids for ids in groups.values() if len(ids) > 1]
        cross = [ids for ids in dup if len({rows[i]['tap'] for i in ids}) > 1]
        conflicts = 0
        for ids in dup:
            signatures = {json.dumps(sorted(rows[i]['khung']), separators=(',', ':')) for i in ids}
            conflict = key == 'pixel_sha256' and len(signatures) > 1
            conflicts += int(conflict)
            cross_split = len({rows[j]['tap'] for j in ids}) > 1
            for i in ids:
                details.append({'loai': key, 'nhom': rows[ids[0]][key], 'anh': rows[i]['anh'],
                                'tap': rows[i]['tap'], 'so_anh_nhom': len(ids),
                                'qua_nhieu_tap': cross_split,
                                'khac_nhan': conflict})
            if key != 'dhash':
                for i in ids[1:]:
                    union(ids[0], i)
        summary.append({'loai': key, 'so_nhom_lap': len(dup), 'so_anh_trong_nhom': sum(map(len, dup)),
                        'so_nhom_qua_nhieu_tap': len(cross), 'so_nhom_pixel_khac_nhan': conflicts})
    components = defaultdict(list)
    for i in range(len(rows)):
        components[root(i)].append(i)
    cross_components = [ids for ids in components.values() if len({rows[i]['tap'] for i in ids}) > 1]
    for ids in components.values():
        name = min(rows[i]['anh'] for i in ids)
        cross_split = len({rows[j]['tap'] for j in ids}) > 1
        for i in ids:
            rows[i]['nhom_lien_quan'] = name
            rows[i]['nhom_qua_nhieu_tap'] = cross_split
    return summary, details, len(cross_components)


def y1_aggregate(rows, total, orphan):
    split, sizes, box_rows, reasons = [], Counter(), [], Counter()
    for r in rows:
        if r['loi_anh']:
            reasons['Ảnh không đọc được'] += 1
        if r['loi_nhan']:
            reasons['Ảnh có nhãn lỗi hoặc thiếu'] += 1
        if not r['loi_nhan'] and r['so_khung'] == 0:
            reasons['Ảnh có tệp nhãn rỗng — cần xác nhận ảnh nền'] += 1
        if r['khung_lap']:
            reasons['Ảnh có dòng khung lặp'] += 1
        if r.get('exif_huong', 1) != 1:
            reasons['Ảnh có thông tin hướng EXIF khác mặc định'] += 1
        if not r['loi_anh']:
            sizes[(r['rong'], r['cao'])] += 1
            for b in r['khung']:
                bw, bh = b[3]*r['rong'], b[4]*r['cao']
                box_rows.append({'anh': r['anh'], 'tap': r['tap'], 'lop': b[0],
                                 'rong_px': bw, 'cao_px': bh, 'ti_le_rong_cao': bw/bh,
                                 'ti_le_dien_tich': b[3]*b[4],
                                 'cao_sau_letterbox_640': bh*min(640/r['rong'], 640/r['cao'])})
    for s in ['train', 'val', 'test']:
        rs = [r for r in rows if r['tap'] == s]
        split.append({'tap': s, 'so_anh_nguon': sum(x['tap'] == s for x in total),
                      'so_anh_da_xet': len(rs), 'anh_doc_duoc': sum(not r['loi_anh'] for r in rs),
                      'anh_loi_nhan': sum(bool(r['loi_nhan']) for r in rs),
                      'anh_nhan_rong': sum(not r['loi_nhan'] and r['so_khung'] == 0 for r in rs),
                      'so_khung_hop_le_da_doc': sum(r['so_khung'] for r in rs)})
    size_rows = [{'rong': w, 'cao': h, 'kich_thuoc': f'{w}x{h}', 'so_anh': n,
                  'ty_le_phan_tram': round(100*n/max(1, sum(sizes.values())), 3)}
                 for (w, h), n in sorted(sizes.items(), key=lambda x: (-x[1], x[0]))]
    reasons['Tệp nhãn không có ảnh tương ứng (toàn ZIP)'] = len(orphan)
    return split, size_rows, box_rows, [{'noi_dung': k, 'so_luong': v} for k, v in reasons.items()]


def y1_load_parts(folder, selected, batch_size):
    rows = []
    for start in range(0, len(selected), batch_size):
        path = folder / f'phan_{start//batch_size:05d}.jsonl.gz'
        if not path.is_file():
            break
        with gzip.open(path, 'rt', encoding='utf-8') as f:
            batch = [json.loads(line) for line in f]
        expect = selected[start:start+batch_size]
        if [r['anh'] for r in batch] != [r['anh'] for r in expect]:
            raise ValueError('Phần tiến độ không khớp danh sách ảnh; không tự bỏ qua.')
        rows.extend(batch)
    return rows


def y1_analyse(project, source, mode='xem_bao_cao', limit=300, batch_size=2000, local_root='/content/yolo1_tam', workers=None):
    if mode not in {'xem_bao_cao', 'xu_ly_du_lieu'} or limit < 0 or batch_size < 1:
        raise ValueError('Chế độ hoặc số lượng không hợp lệ.')
    project, source = Path(project), Path(source)
    scope = 'Toan_bo' if limit == 0 else f'Thu_{limit}_anh'
    report = project / '03_Bao_cao/02_YOLO1' / scope
    if mode == 'xem_bao_cao':
        path = report / 'ket_qua.json'
        if not path.is_file():
            return {'thong_bao': f'Chưa có báo cáo {scope}. Chọn xử lý dữ liệu khi muốn chạy.', 'bao_cao': str(report)}
        result = json.loads(path.read_text(encoding='utf-8'))
        if result.get('phien_ban') != Y1_VERSION or result.get('nguon') != str(source) or result.get('gioi_han') != limit:
            return {'thong_bao': 'Báo cáo không khớp phiên bản mã/đường dẫn/quy mô. Cần chạy lại xử lý.', 'bao_cao': str(report)}
        result['cach_doc'] = 'Đọc báo cáo đã lưu; không mở ZIP và chưa kiểm tra nguồn có thay đổi hay không.'
        return result
    workers = max(1, min(4, os.cpu_count() or 1)) if workers is None else int(workers)
    if workers < 1 or workers > 4:
        raise ValueError('Số luồng cần từ 1 đến 4.')
    t0 = time.monotonic()
    report.mkdir(parents=True, exist_ok=True)
    if not source.is_file():
        raise FileNotFoundError(source)
    stat_before = source.stat()
    with ZipFile(source) as z:
        y1_source_identity(z)
        images, orphan, meta_hash = y1_index(z)
        configs = [i for i in z.infolist() if PurePosixPath(i.filename).name == 'data.yaml']
        if len(configs) != 1 or configs[0].file_size > 1024*1024:
            raise ValueError('Cần đúng một data.yaml nhỏ để xác nhận lớp nguồn.')
        import yaml
        cfg = yaml.safe_load(z.read(configs[0]).decode('utf-8-sig'))
        names = cfg.get('names', [])
        if isinstance(names, dict):
            names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
        if not isinstance(names, list) or len(names) != 1 or int(cfg.get('nc', len(names))) != len(names):
            raise ValueError('YOLO1 cần xác nhận nguồn có đúng một lớp biển số. data.yaml hiện không khớp.')
    t_index = time.monotonic() - t0
    if not images:
        raise ValueError('Không tìm được ảnh trong các tập YOLO.')
    # Mẫu thử phân bố đều theo thứ hạng hash, không chỉ lấy đầu ZIP hay toàn test.
    selected = sorted(images, key=lambda x: hashlib.sha256(x['anh'].encode()).hexdigest())[:limit or None]
    selected.sort(key=lambda x: (x['tap'], x['anh']))
    identity = {'phien_ban': Y1_VERSION, 'thu_vien_giai_ma': y1_decode_versions(), 'metadata_zip': meta_hash, 'duong_dan': str(source),
                'size': stat_before.st_size, 'mtime_ns': stat_before.st_mtime_ns,
                'gioi_han': limit, 'kich_thuoc_phan': batch_size}
    parts = report / 'Tien_do'
    parts.mkdir(exist_ok=True)
    identity_path = parts / 'nguon_va_quy_tac.json'
    if identity_path.is_file():
        if json.loads(identity_path.read_text()) != identity:
            raise ValueError('Nguồn/quy tắc thay đổi so với tiến độ. Giữ bản báo cáo cũ và đổi tên thư mục đợt cũ trước khi chạy mới.')
    else:
        y1_json(identity_path, identity)
    rows = y1_load_parts(parts, selected, batch_size)
    resumed = len(rows)
    print(f'{scope}: {len(selected):,}/{len(images):,} ảnh. Đã lưu: {resumed:,}.', flush=True)
    local = Path(local_root)
    local.mkdir(parents=True, exist_ok=True)
    direct_trial = limit > 0 and len(selected) <= 1000
    cached = source if direct_trial else local / 'Global_v13.zip'
    copied = 0.
    if len(rows) < len(selected) and not direct_trial:
        mark = local / 'nguon.json'
        cache_valid = cached.is_file() and mark.is_file() and cached.stat().st_size == stat_before.st_size and json.loads(mark.read_text()) == identity
        if not cache_valid:
            if shutil.disk_usage(local).free < stat_before.st_size + 512*1024*1024:
                raise OSError('Ổ tạm chưa đủ chỗ cho ZIP nguồn và 512 MiB dự phòng. Chưa sao chép.')
            tc = time.monotonic()
            temp = cached.with_suffix('.part')
            shutil.copyfile(source, temp)
            after = source.stat()
            if (after.st_size, after.st_mtime_ns) != (stat_before.st_size, stat_before.st_mtime_ns) or temp.stat().st_size != stat_before.st_size:
                temp.unlink(missing_ok=True)
                raise ValueError('Nguồn thay đổi trong lúc sao chép; chưa dùng bản sao.')
            temp.replace(cached)
            y1_json(mark, identity)
            copied = time.monotonic() - tc
        print('Đọc ảnh từ một ZIP trên ổ tạm; không giải nén hàng loạt lên Drive.', flush=True)
    if direct_trial and len(rows) < len(selected):
        print('Mẫu thử nhỏ: đọc ảnh đã chọn từ ZIP trên Drive, tránh sao chép cả ZIP lớn.', flush=True)
    scan_t0 = time.monotonic()
    with ZipFile(cached) if len(rows) < len(selected) else io.StringIO() as z:
        for start in range(len(rows), len(selected), batch_size):
            items = selected[start:start+batch_size]
            if workers == 1:
                batch = [y1_one(z, item, len(names)) for item in items]
            else:
                with ThreadPoolExecutor(max_workers=workers) as pool:
                    batch = list(pool.map(lambda item: y1_one(z, item, len(names)), items))
            # Ghi đầy đủ một phần rồi mới công bố trên Drive; phần dở không được tính là đã xong.
            path = parts / f'phan_{start//batch_size:05d}.jsonl.gz'
            temp = local / (path.name + '.part')
            with gzip.open(temp, 'wt', encoding='utf-8') as f:
                for r in batch:
                    f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + '\n')
            drive_tmp = path.with_suffix(path.suffix + '.part')
            shutil.copyfile(temp, drive_tmp)
            drive_tmp.replace(path)
            rows.extend(batch)
            print(f'Đã kiểm tra và lưu {len(rows):,}/{len(selected):,} ảnh.', flush=True)
    scan_seconds = time.monotonic() - scan_t0
    splits, sizes, boxes, issues = y1_aggregate(rows, images, orphan)
    groups, details, cross = y1_groups(rows)
    y1_csv(report/'anh_va_nhan.csv', [dict(anh=r['anh'], tap=r['tap'], rong=r['rong'], cao=r['cao'],
        so_khung=r['so_khung'], loi_anh=r['loi_anh'], loi_nhan=';'.join(r['loi_nhan']),
        khung_lap=r['khung_lap'], pixel_sha256=r['pixel_sha256'], nhom_ten=r['nhom_ten'],
        nhom_lien_quan=r['nhom_lien_quan'], nhom_qua_nhieu_tap=r['nhom_qua_nhieu_tap']) for r in rows])
    for name, data in [('chia_tap', splits), ('kich_thuoc_anh', sizes), ('kich_thuoc_bien', boxes),
                       ('van_de_can_xem', issues), ('nhom_lap', groups), ('chi_tiet_nhom_lap', details)]:
        y1_csv(report/(name+'.csv'), data)
    y1_csv(report/'nhan_khong_co_anh.csv', [{'nhan': p} for p in orphan])
    # Ghi đầy đủ thông tin chất lượng/khung để vẽ lại mà không đọc ảnh.
    detail_tmp = report/'chi_tiet_anh.jsonl.gz.part'
    with gzip.open(detail_tmp, 'wt', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, allow_nan=False)+'\n')
    detail_tmp.replace(report/'chi_tiet_anh.jsonl.gz')
    after = source.stat()
    if (after.st_size, after.st_mtime_ns) != (stat_before.st_size, stat_before.st_mtime_ns):
        raise ValueError('Nguồn thay đổi trong lúc phân tích; chưa công bố kết quả hoàn tất.')
    complete = len(rows) == len(images)
    conclusion = (f'Đã xét {len(rows):,}/{len(images):,} ảnh Global v13. '
        + ('Đây là lượt xét toàn bộ nguồn.' if complete else 'Đây là mẫu thử; không suy rộng số lỗi/trùng thành số liệu toàn bộ dataset.')
        + f' Có {sum(bool(r["loi_anh"]) for r in rows):,} ảnh không đọc được và '
        + f'{sum(bool(r["loi_nhan"]) for r in rows):,} ảnh thiếu/lỗi nhãn. '
        + f'Có {cross:,} nhóm liên quan qua nhiều tập khi nối theo pixel giống nhau hoặc tên gốc Roboflow. '
        + 'Tên gốc và dHash là dấu hiệu cần đối chiếu; chưa chứng minh danh tính cùng xe/biển. '
        + 'Chưa tự loại mẫu, đổi tập hay xuất bộ huấn luyện. Cần xem nhóm mâu thuẫn và ảnh nghi vấn trước khi chốt cách giữ dữ liệu.')
    if sizes:
        top = sizes[0]
        conclusion += f' Kích thước xuất hiện nhiều nhất là {top["kich_thuoc"]}, chiếm {top["ty_le_phan_tram"]:.2f}% số ảnh đọc được trong lượt này.'
    if boxes:
        small = sum(b['cao_sau_letterbox_640'] < 16 for b in boxes)
        conclusion += f' Có {small:,}/{len(boxes):,} khung biển ({100*small/len(boxes):.2f}%) thấp hơn 16 pixel sau letterbox 640; đây là nhóm biển nhỏ cần xem khi đánh giá mô hình, chưa tự loại.'
    conclusion += ' Số ảnh train có thể bao gồm các biến thể tăng cường, không nên diễn giải thành số biển hoặc số xe độc lập.'
    (report/'ket_luan.md').write_text(conclusion, encoding='utf-8')
    result = {'phien_ban': Y1_VERSION, 'thu_vien_giai_ma': y1_decode_versions(), 'nguon': str(source), 'gioi_han': limit, 'bao_cao': str(report),
              'thoi_diem': datetime.now(timezone(timedelta(hours=7))).isoformat(timespec='seconds'),
              'metadata_zip': meta_hash, 'ten_lop_goc': names, 'toan_bo': complete,
              'so_anh_da_xet': len(rows), 'so_anh_nguon': len(images), 'doc_tiep_tu': resumed, 'so_luong_doc_song_song': workers,
              'chia_tap': splits, 'kich_thuoc_anh': sizes, 'van_de': issues, 'nhom_lap': groups,
              'nhom_lien_quan_qua_tap': cross, 'ket_luan': conclusion,
              'thoi_gian_luot_nay_giay': {'kiem_ke': round(t_index,3), 'sao_chep': round(copied,3),
                                          'doc_anh_va_luu_tien_do': round(scan_seconds,3),
                                          'tong': round(time.monotonic()-t0,3)}}
    y1_json(report/'ket_qua.json', result)
    return result


def y1_representatives(result, project, n_sizes=5, per_size=3, mode='xem_bao_cao'):
    if 'so_anh_da_xet' not in result:
        return []
    report = Path(result['bao_cao'])
    path = report/'anh_dai_dien.json'
    if mode == 'xem_bao_cao':
        return json.loads(path.read_text()) if path.is_file() else []
    if mode != 'xu_ly_du_lieu':
        raise ValueError('Chế độ không hợp lệ.')
    with gzip.open(report/'chi_tiet_anh.jsonl.gz', 'rt', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f]
    folder = Path(project)/'04_Anh_dai_dien/YOLO1/Global_v13'/report.name
    folder.mkdir(parents=True, exist_ok=True)
    manifests = []
    # Chỉ mở ảnh đã chọn để xuất mẫu; không quét lại toàn bộ dataset.
    with ZipFile(result['nguon']) as z:
        _, _, current_hash = y1_index(z)
        if current_hash != result['metadata_zip']:
            raise ValueError('Nguồn đã thay đổi; chưa xuất ảnh từ báo cáo cũ.')
        for sz in result['kich_thuoc_anh'][:n_sizes]:
            candidates = [r for r in rows if r['rong'] == sz['rong'] and r['cao'] == sz['cao']
                          and not r['loi_anh'] and not r['loi_nhan']]
            candidates.sort(key=lambda r: hashlib.sha256(r['anh'].encode()).hexdigest())
            chosen, used_pixels, used_families = [], set(), set()
            for r in candidates:
                family = r['nhom_lien_quan']
                if r['pixel_sha256'] in used_pixels or family in used_families:
                    continue
                chosen.append(r)
                used_pixels.add(r['pixel_sha256'])
                used_families.add(family)
                if len(chosen) >= per_size:
                    break
            target = folder/sz['kich_thuoc']
            target.mkdir(exist_ok=True)
            for i, r in enumerate(chosen, 1):
                dest = target / f'{i:02d}_{r["tap"]}{PurePosixPath(r["anh"]).suffix.lower()}'
                raw = z.read(r['anh'])
                dest.write_bytes(raw)
                manifests.append({'kich_thuoc': sz['kich_thuoc'], 'so_anh_nhom': sz['so_anh'],
                    'ty_le_nhom': sz['ty_le_phan_tram'], 'anh_goc': r['anh'], 'tap': r['tap'],
                    'tep_mau': str(dest), 'sha256_tep_mau': hashlib.sha256(raw).hexdigest()})
    y1_csv(folder/'danh_sach_anh_mau.csv', manifests)
    (folder/'Giai_thich.txt').write_text('Ảnh nguyên bản theo kích thước phổ biến trong '
        + ('toàn nguồn.' if result['toan_bo'] else 'mẫu thử; chưa đại diện cho toàn nguồn.')
        + '\nTên tập gốc và đường dẫn ảnh nguồn nằm trong danh_sach_anh_mau.csv.\n'
        + 'Mỗi kích thước lấy tối đa 3 ảnh khác pixel và khác nhóm tên gốc đã nhận diện.\n', encoding='utf-8')
    y1_json(path, manifests)
    return manifests

"""Hiển thị bảng/hình từ báo cáo; không mở nguồn trong chế độ xem."""
def y1_show(result, section):
    from IPython.display import display, Markdown, Image as DisplayImage
    import pandas as pd
    import matplotlib.pyplot as plt
    if 'so_anh_da_xet' not in result:
        print(result.get('thong_bao', 'Chưa có kết quả YOLO1.'))
        return
    report = Path(result['bao_cao'])
    def table(rows, mapping=None):
        frame = pd.DataFrame(rows)
        if mapping:
            frame = frame.rename(columns=mapping)
        display(frame)
    scope = 'TOÀN BỘ NGUỒN' if result['toan_bo'] else 'MẪU THỬ — CHƯA PHẢI THỐNG KÊ TOÀN BỘ'
    if section == 'quy_mo':
        display(Markdown(f'**{scope}** · Đã xét **{result["so_anh_da_xet"]:,} / {result["so_anh_nguon"]:,} ảnh**.'))
        print('Kết quả lưu:', result['thoi_diem'])
        print(result.get('cach_doc', 'Đã xử lý và lưu kết quả.'))
        print('Tên lớp trong nguồn:', ', '.join(map(str, result['ten_lop_goc'])))
        table(result['chia_tap'], {'tap':'Tập', 'so_anh_nguon':'Ảnh trong nguồn', 'so_anh_da_xet':'Ảnh đã xét',
            'anh_doc_duoc':'Ảnh đọc được', 'anh_loi_nhan':'Ảnh thiếu/lỗi nhãn', 'anh_nhan_rong':'Nhãn rỗng',
            'so_khung_hop_le_da_doc':'Khung đúng định dạng'})
        display(Markdown('Số ảnh nguồn đếm từ toàn ZIP; các cột kiểm tra chỉ thuộc những ảnh đã xét. '
            'Một ảnh có thể có nhiều biển. Khung đúng định dạng chưa đồng nghĩa vị trí gán nhãn chính xác.'))
        print('Thời gian lượt xử lý đã lưu (giây):' if result.get('cach_doc') else 'Thời gian lượt vừa xử lý (giây):', result['thoi_gian_luot_nay_giay'])
    elif section == 'kich_thuoc':
        needed = [report/'kich_thuoc_bien.csv', report/'chi_tiet_anh.jsonl.gz']
        missing = [p.name for p in needed if not p.is_file()]
        if missing:
            print('Chưa đủ tệp để vẽ lại phần kích thước/chất lượng:', ', '.join(missing))
            print('Các bảng tổng hợp khác vẫn có thể đọc từ báo cáo đã lưu.')
            return
        table(result['kich_thuoc_anh'][:10], {'kich_thuoc':'Kích thước', 'so_anh':'Số ảnh',
            'ty_le_phan_tram':'Tỷ lệ (%)', 'rong':'Rộng', 'cao':'Cao'})
        boxes = pd.read_csv(report/'kich_thuoc_bien.csv')
        if 'cao_px' in boxes and not boxes.empty:
            display(boxes[['rong_px','cao_px','ti_le_rong_cao','ti_le_dien_tich','cao_sau_letterbox_640']]
                    .describe(percentiles=[.1,.5,.9]).round(3).rename(index={'count':'Số khung','mean':'Trung bình','std':'Độ lệch chuẩn','min':'Nhỏ nhất','max':'Lớn nhất'}, columns={'rong_px':'Rộng biển (px)',
                        'cao_px':'Cao biển (px)', 'ti_le_rong_cao':'Rộng/cao', 'ti_le_dien_tich':'Diện tích/tổng ảnh',
                        'cao_sau_letterbox_640':'Cao khi letterbox 640'}))
            counts = [int((boxes['cao_sau_letterbox_640'] < 16).sum()),
                      int(((boxes['cao_sau_letterbox_640'] >= 16)&(boxes['cao_sau_letterbox_640'] < 32)).sum()),
                      int((boxes['cao_sau_letterbox_640'] >= 32).sum())]
            fig, ax = plt.subplots(figsize=(6,3))
            ax.bar(['< 16 px','16–<32 px','≥ 32 px'], counts, color=['#e9a458','#65a3c4','#528f6c'])
            ax.set(ylabel='Số khung biển', title='Chiều cao biển sau letterbox 640')
            fig.tight_layout()
            display(fig)
            plt.close(fig)
            display(Markdown('Các mốc 16/32 px dùng mô tả biển nhỏ sau khi thu ảnh; chưa dùng để tự loại dữ liệu. '
                'Phép tính giả định letterbox giữ tỷ lệ ảnh, không phải kéo giãn.'))
        with gzip.open(report/'chi_tiet_anh.jsonl.gz','rt',encoding='utf-8') as f:
            rows = [json.loads(line) for line in f]
        quality = [r['chat_luong_bien_lon_nhat'] for r in rows if r.get('chat_luong_bien_lon_nhat') and not r['loi_anh']]
        if quality:
            display(Markdown('**Đặc điểm vùng biển lớn nhất trong mỗi ảnh** — mức sáng và độ tương phản theo thang 0–255; '
                'độ nét đo sau khi thu vùng ảnh về cạnh tối đa 256 px. Chưa áp dụng ngưỡng loại.'))
            display(pd.DataFrame(quality).describe().round(2).rename(index={'count':'Số ảnh','mean':'Trung bình','std':'Độ lệch chuẩn','min':'Nhỏ nhất','max':'Lớn nhất'}, columns={'sang':'Mức sáng', 'tuong_phan':'Tương phản','net':'Chỉ số nét'}))
    elif section == 'kiem_tra':
        table(result['van_de'], {'noi_dung':'Nội dung cần xem', 'so_luong':'Số lượng'})
        human_names = {'pixel_sha256':'Ảnh giống toàn bộ pixel', 'nhom_ten':'Cùng tên gốc Roboflow', 'dhash':'Gợi ý ảnh tương tự (dHash)'}
        groups = [dict(r, loai=human_names.get(r['loai'],r['loai'])) for r in result['nhom_lap']]
        table(groups, {'loai':'Cách đối chiếu', 'so_nhom_lap':'Nhóm lặp',
            'so_anh_trong_nhom':'Ảnh trong nhóm', 'so_nhom_qua_nhieu_tap':'Nhóm qua nhiều tập',
            'so_nhom_pixel_khac_nhan':'Nhóm pixel giống nhưng khác nhãn'})
        display(Markdown('**Cách đọc:** Nhóm giống pixel so sánh toàn bộ pixel RGB và kích thước; '
            'nhóm tên gốc nối ảnh có cùng phần tên trước `.rf.`; dHash chỉ gợi ý ảnh có cấu trúc sáng/tối giống nhau. '
            'dHash trùng không đủ kết luận ảnh trùng. Các hàng có thể chồng lặp, không cộng thành tổng số phải bỏ. '
            'Khác nhãn ở đây là khác trị số của khung; cần xem độ lệch và ảnh trước khi kết luận gán nhãn sai. Ảnh có nhãn rỗng cần kiểm tra có đúng là ảnh nền. Chưa tự sửa nhãn, loại ảnh hoặc chuyển tập.'))
        print('Chi tiết từng ảnh/nhóm được lưu trong:', report)
    elif section == 'ket_luan':
        display(Markdown(result['ket_luan']))
        print('Báo cáo:', report)


def y1_preview(result, manifest, mode='xem_bao_cao'):
    from IPython.display import display, Image as DisplayImage
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    if not manifest:
        return
    path = Path(result['bao_cao'])/'anh_mau_co_khung.png'
    if mode == 'xu_ly_du_lieu':
        wanted = {r['anh_goc'] for r in manifest}
        with gzip.open(Path(result['bao_cao'])/'chi_tiet_anh.jsonl.gz','rt',encoding='utf-8') as f:
            rows = {r['anh']:r for r in map(json.loads,f) if r['anh'] in wanted}
        nr = math.ceil(len(manifest)/3)
        fig, axes = plt.subplots(nr,3,figsize=(13,3.5*nr),squeeze=False)
        for ax, rep in zip(axes.flat, manifest):
            with Image.open(rep['tep_mau']) as im:
                ax.imshow(im.convert('RGB'))
                w,h=im.size
            for b in rows[rep['anh_goc']]['khung']:
                ax.add_patch(Rectangle(((b[1]-b[3]/2)*w,(b[2]-b[4]/2)*h),b[3]*w,b[4]*h,
                                       fill=False,edgecolor='#00ff5e',linewidth=1.5))
            ax.set_title(rep['kich_thuoc']+' · '+rep['tap'],fontsize=10)
            ax.axis('off')
        for ax in list(axes.flat)[len(manifest):]:
            ax.axis('off')
        fig.tight_layout()
        fig.savefig(path,dpi=120,bbox_inches='tight')
        plt.close(fig)
    if not path.is_file():
        print('Chưa có hình kiểm tra đã lưu cho lượt này.')
        return
    display(DisplayImage(filename=str(path), width=1050))
    print('Hình kiểm tra có khung nằm trong báo cáo. Thư mục ảnh nộp chứa ảnh gốc không vẽ khung.')


In [ ]:
# @title Chạy thử YOLO1 hoặc mở báo cáo đã lưu
SO_ANH_THU_YOLO1 = 0 # @param {type:"integer"}
# 300: thử trước; 0: toàn bộ Global v13. Giữ nguyên số này khi xem lại cùng báo cáo.
NGUON_YOLO1 = TEP_GLOBAL or str(Path(THU_MUC_DU_AN)/"01_Du_lieu_goc/Global_v13/license-plate-dataset.zip")
Y1_KET_QUA = {'thong_bao':'Lượt YOLO1 hiện tại chưa hoàn tất.'}
Y1_KET_QUA = y1_analyse(THU_MUC_DU_AN, NGUON_YOLO1, CHE_DO, SO_ANH_THU_YOLO1)
y1_show(Y1_KET_QUA, 'quy_mo')


### 2.2. Kích thước ảnh và biển số có đặc điểm gì?

Ảnh đầu vào cùng kích thước chưa chắc có biển số lớn như nhau. Bảng kích thước ảnh cho biết những nhóm phổ biến để chọn mẫu đại diện; bảng vùng biển và biểu đồ cho biết biển còn cao bao nhiêu pixel khi đưa ảnh về khung 640 × 640 và giữ nguyên tỷ lệ. Các số liệu này mô tả dữ liệu, chưa phải ngưỡng loại ảnh.


In [ ]:
# @title Xem kích thước và đặc điểm vùng biển
y1_show(Y1_KET_QUA, 'kich_thuoc')


### 2.3. Có lỗi hoặc ảnh liên quan nằm ở nhiều tập không?

Chúng ta đọc từng cặp ảnh–nhãn, ghi riêng ảnh lỗi, nhãn thiếu, nhãn rỗng, khung sai và khung lặp. Sau đó đối chiếu toàn bộ pixel, nhóm tên ảnh Roboflow và dấu hiệu hình ảnh tương tự. Nếu các ảnh liên quan nằm ở cả train và tập đánh giá, cần xem lại để tránh đánh giá quá lạc quan.

Tên gốc Roboflow là một dấu hiệu về ảnh tăng cường; chưa đủ để biết chắc mọi ảnh cùng xe hoặc biển. Mã không tự xóa các trường hợp nghi vấn. Trước khi chốt cách giữ dữ liệu, cần xem ảnh và khung của những nhóm này, đặc biệt nhóm giống pixel nhưng khác nhãn.


In [ ]:
# @title Xem các vấn đề và nhóm ảnh cần đối chiếu
y1_show(Y1_KET_QUA, 'kiem_tra')


### 2.4. Trích xuất ảnh đại diện để kiểm tra và nộp

Với tối đa năm kích thước xuất hiện nhiều nhất trong lượt đã xét, lấy tối đa ba ảnh khác pixel và khác nhóm tên đã nhận diện. Ảnh được giữ nguyên nội dung gốc, kèm bảng đường dẫn nguồn, tập và tỷ lệ của nhóm kích thước.

Vị trí: `04_Anh_dai_dien/YOLO1/Global_v13/Thu_300_anh/<rộng>x<cao>/` cho lượt thử 300; nhánh `Toan_bo` dành riêng cho lượt đầy đủ. Mẫu thử chưa được gọi là mẫu đại diện cho toàn bộ nguồn.


In [ ]:
# @title Trích xuất hoặc xem danh sách ảnh đại diện YOLO1
import pandas as pd
from IPython.display import display
Y1_ANH_MAU = y1_representatives(Y1_KET_QUA, THU_MUC_DU_AN, mode=CHE_DO)
if Y1_ANH_MAU:
    display(pd.DataFrame(Y1_ANH_MAU)[['kich_thuoc','so_anh_nhom','ty_le_nhom','anh_goc','tap','tep_mau']].rename(columns={
        'kich_thuoc':'Kích thước','so_anh_nhom':'Số ảnh trong nhóm','ty_le_nhom':'Tỷ lệ nhóm (%)',
        'anh_goc':'Ảnh trong ZIP nguồn','tap':'Tập gốc','tep_mau':'Ảnh đã xuất'}))
    print('Mẫu thử' if not Y1_KET_QUA['toan_bo'] else 'Toàn nguồn', '— giữ nguyên byte ảnh gốc, không vẽ khung lên ảnh nộp.')
else:
    print('Chưa có ảnh mẫu đã lưu cho lượt này. Chế độ xem không tự trích xuất ảnh.')
y1_preview(Y1_KET_QUA, Y1_ANH_MAU, CHE_DO)


### 2.5. Kết luận và việc cần xác nhận

Kết luận được tạo từ báo cáo của đúng lượt chạy. Phần này nêu phạm vi đã xét và những vấn đề cần xem tiếp; chưa gọi số ảnh đọc được là số ảnh huấn luyện cuối cùng. Sau khi kiểm tra ảnh nghi vấn, chúng ta mới chốt quy tắc xử lý trùng và xuất bộ YOLO1.


In [ ]:
# @title Kết luận phân tích YOLO1
y1_show(Y1_KET_QUA, 'ket_luan')


### 2.6. Kiểm tra những ảnh cần xem lại

Sau thống kê toàn nguồn, chúng ta xem riêng các trường hợp cần quyết định: ảnh trùng pixel nhưng khác khung, khung sai quy tắc, nhãn rỗng và ảnh có cấu trúc tương tự. Ảnh được lấy đúng từ nguồn đã phân tích; bảng bên dưới ghi mã để tìm lại từng ảnh.

**Đây là bước rà soát và lập phương án.** Các ảnh gốc và cách chia nguồn vẫn được giữ nguyên. Phương án ưu tiên test, rồi validation, rồi train khi một nhóm pixel/tên gốc xuất hiện ở nhiều tập; mọi biến thể train của nhóm đó được xét cùng nhau. Ảnh có nhãn lỗi hoặc nhãn rỗng chưa xác nhận được ghi riêng để xem xét. Các gợi ý dHash chưa được dùng để tự loại.


In [ ]:
# @title Các hàm tạo bảng và ảnh rà soát — có thể thu gọn
"""Rà soát từ metadata YOLO1; chỉ tạo báo cáo và phương án, không sửa dữ liệu nguồn."""
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from zipfile import ZipFile
from PIL import Image, ImageOps, ImageDraw, ImageFont
import gzip, json, math, time

Y1_REVIEW_VERSION = 'ra_soat_yolo1_1.1'
Y1_PRIORITY = {'train':0, 'val':1, 'test':2}


def y1_review_plan(rows):
    strong = defaultdict(list)
    for r in rows:
        strong[r['nhom_lien_quan']].append(r)
    cross = {k:v for k,v in strong.items() if len({r['tap'] for r in v})>1}
    top = {k:max(Y1_PRIORITY[r['tap']] for r in rs) for k,rs in cross.items()}
    dh = defaultdict(list)
    for r in rows:
        if r['dhash']:
            dh[r['dhash']].append(r)
    weak = {}
    for k,rs in dh.items():
        if len({r['tap'] for r in rs})>1 and len({r['nhom_lien_quan'] for r in rs})>1:
            weak[k]=rs
    proposal=[]
    for r in rows:
        flags=[]
        if r['loi_anh']:
            flags.append('anh_khong_doc_duoc')
        if r['loi_nhan']:
            flags.append('nhan_khong_hop_le')
        elif r['so_khung']==0:
            flags.append('can_xac_nhan_anh_nen')
        g=r['nhom_lien_quan']
        if g in top and Y1_PRIORITY[r['tap']]<top[g]:
            flags.append('trung_nhom_voi_tap_uu_tien_cao_hon')
        proposal.append({'anh':r['anh'],'tap_goc':r['tap'], 'nhom_lien_quan':g,
                         'de_xuat_giu':not flags,'ly_do_tam_giu_ngoai':';'.join(flags)})
    before,after=Counter(r['tap'] for r in rows), Counter(r['tap_goc'] for r in proposal if r['de_xuat_giu'])
    table=[{'tap':s,'ban_dau':before[s],'de_xuat_giu':after[s], 'tam_giu_ngoai':before[s]-after[s]}
           for s in ['train','val','test']]
    # Kiểm tra phương án chỉ giữ một tập trong từng nhóm đã nối theo pixel/tên gốc.
    kept=defaultdict(set)
    for r in proposal:
        if r['de_xuat_giu']:
            kept[r['nhom_lien_quan']].add(r['tap_goc'])
    if any(len(s)>1 for s in kept.values()):
        raise AssertionError('Phương án vẫn có nhóm pixel/tên gốc ở nhiều tập.')
    # dHash vẫn chỉ là gợi ý; không dùng để tự loại hoặc nối thêm nhóm ở phương án này.
    return proposal,table,cross,weak


def y1_review_label_numbers(text):
    boxes=[]
    for line in text.splitlines():
        parts=line.split()
        try:
            vals=list(map(float,parts))
            if len(vals)==5 and all(math.isfinite(x) for x in vals):
                boxes.append(vals)
        except ValueError:
            pass
    return boxes


def y1_review_sheet(z,cases,target,columns=3,per_page=9):
    # Mỗi ô: ảnh thu nhỏ + mã để tìm đúng tệp trong CSV, tối đa hai bộ khung trên cùng ảnh.
    try:
        font=ImageFont.truetype('DejaVuSans.ttf',14)
    except OSError:
        font=ImageFont.load_default()
    tw,th=380,270
    pages=[]
    for offset in range(0,len(cases),per_page):
        group=cases[offset:offset+per_page]
        nr=math.ceil(len(group)/columns)
        canvas=Image.new('RGB',(columns*tw,nr*th),'white')
        d=ImageDraw.Draw(canvas)
        for j,case in enumerate(group):
            row=case['members'][0]
            with Image.open(__import__('io').BytesIO(z.read(row['anh']))) as im:
                im=im.convert('RGB')
                ow,oh=im.size
                im.thumbnail((tw-12,th-62))
            x=(j%columns)*tw+6
            y=(j//columns)*th+44
            canvas.paste(im,(x,y))
            sx,sy=im.width/ow,im.height/oh
            draw=ImageDraw.Draw(canvas)
            for mi,member in enumerate(case['members']):
                color=['#00dd55','#ee6100','#9560ff'][mi%3]
                boxes=member.get('raw_boxes',member['khung'])
                for b in boxes:
                    _,cx,cy,bw,bh=b
                    # Clip để vẽ trên ô ảnh; không sửa trị số nhãn gốc.
                    xa=max(0,min(ow,(cx-bw/2)*ow)); xb=max(0,min(ow,(cx+bw/2)*ow))
                    ya=max(0,min(oh,(cy-bh/2)*oh)); yb=max(0,min(oh,(cy+bh/2)*oh))
                    if xb>=xa and yb>=ya:
                        draw.rectangle((x+xa*sx,y+ya*sy,x+xb*sx,y+yb*sy),outline=color,width=2)
            title=case['id']+' | '+' / '.join(m['tap'] for m in case['members'])
            draw.text((x,y-40),title,fill='black',font=font)
            draw.text((x,y-21),PurePosixPath(row['anh']).stem[:45],fill='black',font=font)
            draw.text((x,(j//columns)*th+th-17),case.get('note','')[:48],fill='black',font=font)
        page=target.with_name(target.stem+f'_{offset//per_page+1:02d}.jpg')
        canvas.save(page,quality=94)
        pages.append(str(page))
    return pages


def y1_review_build(project,result,mode='xem_bao_cao'):
    if 'so_anh_da_xet' not in result or not result['toan_bo']:
        return {'thong_bao':'Cần báo cáo toàn nguồn YOLO1 trước khi rà soát.'}
    folder=Path(result['bao_cao'])/'Ra_soat'
    summary_path=folder/'ket_qua_ra_soat.json'
    if mode=='xem_bao_cao':
        if not summary_path.is_file():
            return {'thong_bao':'Chưa có báo cáo rà soát đã lưu. Chế độ xem không tự tạo ảnh.'}
        summary=json.loads(summary_path.read_text())
        if summary.get('metadata_zip')!=result['metadata_zip'] or summary.get('phien_ban')!=Y1_REVIEW_VERSION:
            return {'thong_bao':'Báo cáo rà soát chưa khớp nguồn/phiên bản.'}
        return summary
    if mode!='xu_ly_du_lieu':
        raise ValueError('Chế độ không hợp lệ.')
    start=time.monotonic()
    with gzip.open(Path(result['bao_cao'])/'chi_tiet_anh.jsonl.gz','rt',encoding='utf-8') as f:
        rows=[json.loads(x) for x in f]
    if len(rows)!=result['so_anh_da_xet']:
        raise ValueError('Số bản ghi không khớp báo cáo phân tích.')
    proposal,table,cross,weak=y1_review_plan(rows)
    folder.mkdir(exist_ok=True)
    y1_csv(folder/'phuong_an_giu_du_lieu.csv',proposal)
    y1_csv(folder/'so_luong_theo_phuong_an.csv',table)
    pixels=defaultdict(list)
    for r in rows:
        if r['pixel_sha256']:
            pixels[r['pixel_sha256']].append(r)
    dup=[v for v in pixels.values() if len(v)>1 and len({r['tap'] for r in v})>1]
    dup.sort(key=lambda v:min(r['anh'] for r in v))
    bad=[r for r in rows if r['loi_nhan']]
    empty=[r for r in rows if not r['loi_nhan'] and r['so_khung']==0]
    items={'Anh_trung_pixel':[], 'Nhan_can_xem':[], 'Nhan_rong':[], 'Goi_y_anh_tuong_tu':[], 'Nhom_cung_ten':[]}
    index=[]
    for i,rs in enumerate(dup,1):
        members=sorted(rs,key=lambda r:(-Y1_PRIORITY[r['tap']],r['anh']))
        items['Anh_trung_pixel'].append({'id':f'P{i:03d}','members':members,'note':'Xanh: tap uu tien; cam: nhan con lai'})
    _pixel_cross_components={r['nhom_lien_quan'] for rs in dup for r in rs}
    _name_only=[rs for g,rs in sorted(cross.items()) if g not in _pixel_cross_components]
    for i,rs in enumerate(_name_only,1):
        for j,split in enumerate(['test','val','train'],1):
            candidates=[r for r in rs if r['tap']==split]
            if candidates:
                r=max(candidates,key=lambda r:(r['rong'] or 0)*(r['cao'] or 0))
                items['Nhom_cung_ten'].append({'id':f'R{i:03d}-{j}','members':[r],
                    'note':f'Cung nhom R{i:03d}; chua co pixel trung qua tap'})
    for i,r in enumerate(bad,1):
        items['Nhan_can_xem'].append({'id':f'L{i:03d}','members':[r],'note':'Khung goc chi clip khi ve hinh'})
    for i,r in enumerate(empty,1):
        items['Nhan_rong'].append({'id':f'E{i:03d}','members':[r],'note':'Can xem co bien chua duoc gan nhan khong'})
    # Lấy một ảnh đại diện cho MỖI nhóm gốc trong toàn bộ gợi ý dHash qua tập để đối chiếu.
    weak_items=sorted(weak.items(),key=lambda kv:(-len({r['nhom_lien_quan'] for r in kv[1]}),kv[0]))
    for i,(hash_value,rs) in enumerate(weak_items,1):
        representatives={}
        for r in sorted(rs,key=lambda r:(-Y1_PRIORITY[r['tap']],r['anh'])):
            representatives.setdefault(r['nhom_lien_quan'],r)
        picked=list(representatives.values())
        # dHash khác pixel nên mỗi ảnh cần một ô riêng để so sánh, không chồng khung lên ảnh khác.
        for j,r in enumerate(picked,1):
            items['Goi_y_anh_tuong_tu'].append({'id':f'H{i:03d}-{j}','members':[r],
                 'note':f'Cung goi y H{i:03d}; chua ket luan trung'})
    local=Path('/content/yolo1_tam/Global_v13.zip')
    source=local if local.is_file() else Path(result['nguon'])
    bad_numbers=[]
    pages={}
    with ZipFile(source) as z:
        _,_,current_hash=y1_index(z)
        if current_hash!=result['metadata_zip']:
            raise ValueError('ZIP để xem ảnh không khớp nguồn đã phân tích.')
        for case in items['Nhan_can_xem']:
            r=case['members'][0]
            text=z.read(r['nhan']).decode('utf-8-sig') if r['nhan'] else ''
            r['raw_boxes']=y1_review_label_numbers(text)
            for b in r['raw_boxes']:
                _,cx,cy,bw,bh=b
                left,top,right,bottom=cx-bw/2,cy-bh/2,cx+bw/2,cy+bh/2
                bad_numbers.append({'id':case['id'],'anh':r['anh'],'rong_anh':r['rong'],'cao_anh':r['cao'],
                    'cx':cx,'cy':cy,'bw':bw,'bh':bh,'can_trai':left,'can_tren':top,'can_phai':right,'can_duoi':bottom,
                    'do_vuot_bien_lon_nhat_px':max(0,-left*r['rong'],-top*r['cao'],(right-1)*r['rong'],(bottom-1)*r['cao']),
                    'dien_tich_khong_duong':bw<=0 or bh<=0})
        for category,cases in items.items():
            target=folder/category
            target.mkdir(exist_ok=True)
            print(f'Tạo hình {category}: {len(cases)} ô ảnh.',flush=True)
            pages[category]=y1_review_sheet(z,cases,target/'Trang.jpg')
            for case in cases:
                for k,r in enumerate(case['members']):
                    index.append({'loai':category,'id':case['id'],'anh':r['anh'],'tap':r['tap'],
                                  'thu_tu_bo_khung':k,'nhom_lien_quan':r['nhom_lien_quan']})
    y1_csv(folder/'anh_can_doi_chieu.csv',index)
    y1_csv(folder/'tri_so_nhan_can_xem.csv',bad_numbers)
    bad_summary={'so_dong_nhan_da_doc':len(bad_numbers),
        'so_dong_dien_tich_khong_duong':sum(x['dien_tich_khong_duong'] for x in bad_numbers),
        'so_dong_vuot_bien':sum(x['do_vuot_bien_lon_nhat_px']>0 for x in bad_numbers),
        'vuot_bien_lon_nhat_px':max((x['do_vuot_bien_lon_nhat_px'] for x in bad_numbers),default=0)}
    summary={'phien_ban':Y1_REVIEW_VERSION,'metadata_zip':result['metadata_zip'],'thu_muc':str(folder),
        'phuong_an':table,'so_nhom_qua_tap_theo_pixel_ten':len(cross), 'so_nhom_chi_ten_qua_tap':len(_name_only),
        'so_goi_y_dhash_khac_nhom_goc_qua_tap':len(weak),
        'so_goi_y_dhash_da_tao_mau':len(weak_items),
        'so_anh_nhan_rong':len(empty),'so_nhom_ten_cua_anh_nhan_rong':len({r['nhom_lien_quan'] for r in empty}),
        'so_anh_nhan_can_xem':len(bad), 'tri_so_nhan_loi':bad_summary,
        'trang_anh':pages,'thoi_gian_giay':round(time.monotonic()-start,3),
        'luu_y':'Chỉ tạo phương án và ảnh để rà soát. Chưa sửa nhãn, đổi tập hay xuất bộ huấn luyện. dHash không dùng tự loại.'}
    y1_json(summary_path,summary)
    return summary


def y1_review_show(summary,kind='Nhan_can_xem',page=1):
    from IPython.display import display,Image as DisplayImage
    if 'thong_bao' in summary:
        print(summary['thong_bao'])
        return
    paths=summary['trang_anh'].get(kind,[])
    if not paths:
        print('Chưa có trang ảnh cho nhóm này.')
        return
    if not 1<=page<=len(paths):
        print(f'Chọn trang từ 1 đến {len(paths)}.')
        return
    print(kind, f'— trang {page}/{len(paths)}')
    display(DisplayImage(filename=paths[page-1],width=1000))

"""Lập phương án từ các quyết định đã nhìn ảnh; chưa tạo dữ liệu huấn luyện."""
from pathlib import Path
from collections import Counter, defaultdict
import csv, gzip, io, json, uuid
Y1_PRIORITY = {'train':0, 'val':1, 'test':2}

def y1_refine_proposal(result, decisions):
    if decisions.get('phien_ban_bo_anh') != 'ra_soat_yolo1_1.1':
        raise ValueError('Quyết định chưa khớp phiên bản bảng ảnh.')
    if decisions['metadata_zip']!=result['metadata_zip']:
        raise ValueError('Quyết định rà soát không khớp dữ liệu.')
    folder=Path(result['bao_cao'])/'Ra_soat'
    with gzip.open(Path(result['bao_cao'])/'chi_tiet_anh.jsonl.gz','rt',encoding='utf-8') as f:
        rows=[json.loads(x) for x in f]
    with (folder/'anh_can_doi_chieu.csv').open(encoding='utf-8-sig') as f:
        index=list(csv.DictReader(f))
    with (folder/'tri_so_nhan_can_xem.csv').open(encoding='utf-8-sig') as f:
        raw_bad=list(csv.DictReader(f))
    # Các khung đã nhìn ảnh: chỉ vượt mép khoảng một pixel và còn diện tích dương sau clip.
    repair_ids=set(decisions['nhan_vuot_bien']['ids'])
    repair={r['anh'] for r in raw_bad if r['id'] in repair_ids}
    for r in raw_bad:
        if r['anh'] in repair:
            if r['dien_tich_khong_duong']!='False' or float(r['do_vuot_bien_lon_nhat_px'])>1.01:
                raise ValueError('Có khung vượt phạm vi đã rà soát; chưa tự sửa.')
    empty_status={d['id']:d['ket_qua'] for d in decisions['nhan_rong']}
    empty_by_image={r['anh']:empty_status[r['id']] for r in index if r['loai']=='Nhan_rong'}
    parent={r['nhom_lien_quan']:r['nhom_lien_quan'] for r in rows}
    def root(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x
    def union(a,b):
        a,b=root(a),root(b)
        if a!=b:parent[b]=a
    manual_groups=defaultdict(set)
    confirmed=set(decisions['dhash_xac_nhan'])
    for r in index:
        if r['loai']=='Goi_y_anh_tuong_tu':
            key=r['id'].split('-')[0]
            if key in confirmed:manual_groups[key].add(r['nhom_lien_quan'])
    if len(manual_groups)!=len(confirmed):
        raise ValueError('Thiếu ảnh đại diện của nhóm đã xác nhận.')
    for groups in manual_groups.values():
        values=sorted(groups)
        for g in values[1:]:union(values[0],g)
    grouped=defaultdict(list)
    for r in rows:grouped[root(r['nhom_lien_quan'])].append(r)
    proposal=[]
    row_flags={}
    for r in rows:
        flags=[]
        if r['loi_anh']:flags.append('anh_khong_doc_duoc')
        if r['loi_nhan'] and r['anh'] not in repair:flags.append('nhan_loi_chua_xu_ly')
        if not r['loi_nhan'] and r['so_khung']==0:
            st=empty_by_image.get(r['anh'],'chua_xem')
            if st=='co_bien_ro_thieu_nhan':flags.append('bien_ro_nhung_thieu_nhan')
            elif st!='khong_thay_ro_bien_tren_bang_ghep':flags.append('nhan_rong_can_xem_them')
        row_flags[r['anh']]=flags
    chosen_eval={}
    top={g:max(Y1_PRIORITY[r['tap']] for r in rs) for g,rs in grouped.items()}
    for g,rs in grouped.items():
        if top[g]>0:
            candidates=[r for r in rs if Y1_PRIORITY[r['tap']]==top[g] and not row_flags[r['anh']]]
            if candidates:
                # Một đại diện đánh giá/nhóm đã xác nhận; ưu tiên nhãn đầy đủ và ảnh có nhiều pixel.
                candidates.sort(key=lambda r:(-(r['so_khung']+int(r['anh'] in repair)),
                    -(r['rong']*r['cao']),r['anh']))
                chosen_eval[g]=candidates[0]['anh']
    for r in rows:
        g=root(r['nhom_lien_quan'])
        flags=list(row_flags[r['anh']])
        if Y1_PRIORITY[r['tap']]<top[g]:flags.append('nhom_da_co_o_tap_uu_tien_cao_hon')
        elif top[g]>0 and r['anh']!=chosen_eval.get(g):flags.append('chi_mot_dai_dien_danh_gia_moi_nhom')
        proposal.append({'anh':r['anh'],'tap_goc':r['tap'],'nhom_sau_ra_soat':g,
            'de_xuat_giu':not flags, 'de_xuat_chinh_khung':r['anh'] in repair,
            'ly_do_tam_giu_ngoai':';'.join(flags)})
    byname={r['anh']:r for r in rows}
    table=[]
    for s in ['train','val','test']:
        rs=[p for p in proposal if p['tap_goc']==s]
        table.append({'tap':s,'ban_dau':len(rs),'de_xuat_giu':sum(p['de_xuat_giu'] for p in rs),
                      'tam_giu_ngoai':sum(not p['de_xuat_giu'] for p in rs)})
    kept=defaultdict(set)
    for p in proposal:
        if p['de_xuat_giu']:kept[p['nhom_sau_ra_soat']].add(p['tap_goc'])
    assert not any(len(s)>1 for s in kept.values())
    y1_csv(folder/'phuong_an_sau_ra_soat.csv',proposal)
    y1_csv(folder/'so_luong_sau_ra_soat.csv',table)
    y1_json(folder/'quyet_dinh_ra_soat.json',decisions)
    summary={'phien_ban':'phuong_an_sau_ra_soat_1.0','metadata_zip':result['metadata_zip'],
        'phuong_an':table,'so_khung_co_the_chinh_vao_bien':len(repair),
        'so_anh_de_xuat_giu_can_chinh_khung':sum(p['de_xuat_giu'] and p['de_xuat_chinh_khung'] for p in proposal),
        'nhan_rong':dict(Counter(d['ket_qua'] for d in decisions['nhan_rong'])),
        'goi_y_dhash_da_xem':len(decisions['dhash_da_xem_ban_mau']['ids']),
        'goi_y_dhash_xac_nhan':len(confirmed),'goi_y_dhash_chua_ket_luan':decisions['dhash_chua_ket_luan'],
        'so_nhom_da_xac_nhan_qua_tap':sum(len({r['tap'] for r in rs})>1 for rs in grouped.values()),
        'so_nhom_qua_tap_trong_phuong_an':sum(len(v)>1 for v in kept.values()),
        'so_khung_du_kien':sum(byname[p['anh']]['so_khung']+int(p['de_xuat_chinh_khung']) for p in proposal if p['de_xuat_giu']),
        'luu_y':'Đây là phương án từ rà soát bảng ảnh; chưa sửa nhãn gốc hay xuất bộ huấn luyện. H123 chưa được dùng để nối nhóm.'}
    y1_json(folder/'phuong_an_sau_ra_soat.json',summary)
    return summary


In [ ]:
# @title Tạo ảnh rà soát hoặc mở kết quả đã lưu
import pandas as pd
from IPython.display import display
Y1_RA_SOAT = y1_review_build(THU_MUC_DU_AN, Y1_KET_QUA, CHE_DO)
if 'thong_bao' in Y1_RA_SOAT:
    print(Y1_RA_SOAT['thong_bao'])
else:
    _review_path = Path(Y1_RA_SOAT['thu_muc'])
    _decision_path = _review_path/'quyet_dinh_ra_soat.json'
    if CHE_DO == 'xu_ly_du_lieu' and _decision_path.is_file():
        y1_refine_proposal(Y1_KET_QUA, json.loads(_decision_path.read_text(encoding='utf-8')))
    _refined_path = _review_path/'phuong_an_sau_ra_soat.json'
    _review_table = Y1_RA_SOAT['phuong_an']
    _review_notice = 'Phương án ban đầu, chưa có kết luận bằng mắt.'
    if _refined_path.is_file():
        _refined = json.loads(_refined_path.read_text(encoding='utf-8'))
        if _refined['metadata_zip'] == Y1_KET_QUA['metadata_zip']:
            _review_table = _refined['phuong_an']
            _review_notice = 'Bảng đã cập nhật theo ghi nhận nhìn ảnh; chưa xuất dữ liệu huấn luyện.'
    print(_review_notice)
    display(pd.DataFrame(_review_table).rename(columns={
        'tap':'Tập gốc','ban_dau':'Ảnh ban đầu','de_xuat_giu':'Dự kiến giữ','tam_giu_ngoai':'Tạm giữ ngoài để xem'}))
    print('Thư mục rà soát:',_review_path)


In [ ]:
# @title Xem từng trang ảnh cần đối chiếu
LOAI_ANH_RA_SOAT = "Nhan_can_xem" # @param ["Nhan_can_xem","Nhan_rong","Anh_trung_pixel","Goi_y_anh_tuong_tu","Nhom_cung_ten"]
TRANG_RA_SOAT = 1 # @param {type:"integer"}
y1_review_show(Y1_RA_SOAT, LOAI_ANH_RA_SOAT, TRANG_RA_SOAT)


### 2.7. Kết luận sau khi nhìn ảnh và phương án xử lý

Đã xem các bảng ghép của **67 nhóm trùng pixel**, **26 ảnh có khung vượt mép**, **144 ảnh nhãn rỗng**, các ảnh đại diện của **134 gợi ý gần giống** và **5 nhóm chỉ cùng tên gốc**.

- Cả 26 khung vượt biên vẫn nằm ở vùng biển quan sát được; mức vượt lớn nhất khoảng **1,01 pixel**, diện tích dương. Đề xuất chỉnh vào đúng biên ảnh, giữ nguyên ảnh. Trong phương án hiện tại còn 25 ảnh cần chỉnh khung; một ảnh đã nằm trong nhóm tạm giữ ngoài vì lý do khác.
- Nhãn rỗng có **28 ảnh thấy biển rõ nhưng thiếu nhãn**; **56 ảnh không thấy biển trên bảng ghép**; **60 ảnh cần phóng lớn hoặc thống nhất tiêu chí**, gồm cả biển/ốp biển không có ký tự. Đây là ghi nhận từ bảng ảnh, chưa khẳng định đã xác minh mọi ảnh ở độ phân giải gốc.
- Trong 134 gợi ý gần giống đã xem đủ các đại diện, **133 nhóm có quan hệ ảnh/cảnh/xe quan sát được**. Nhóm `H123` có xe cùng dòng nhưng biển và thời gian khác nhau; chưa dùng nhóm này để tự nối hoặc loại ảnh.

Phương án ưu tiên test, rồi validation, rồi train khi các ảnh có quan hệ đã xác nhận. Giữ một đại diện đánh giá trong mỗi nhóm liên quan; giữ các biến thể train nếu nhóm không đi sang tập đánh giá. Các ảnh thiếu nhãn/chưa rõ được ghi riêng. Theo những quan hệ đã kiểm tra, phương án không còn nhóm xác nhận nằm ở nhiều tập; điều này chưa chứng minh mọi xe trong dataset đều độc lập theo danh tính.

**Dự kiến giữ 98.983 ảnh:** 95.941 train, 2.023 validation và 1.019 test. Bảng đếm mỗi ảnh một lần dù một ảnh có thể có nhiều lý do cần giữ riêng. Đây là phương án từ rà soát; **chưa sửa nhãn nguồn hoặc xuất bộ huấn luyện**. Bước tiếp theo là áp dụng phương án và kiểm tra dữ liệu xuất ra.


In [ ]:
# @title Xem biên bản rà soát và vị trí tệp phương án
if 'trang_anh' in Y1_RA_SOAT:
    display(pd.DataFrame([{'Nhóm ảnh':k,'Số trang đã tạo':len(v)} for k,v in Y1_RA_SOAT['trang_anh'].items()]))
    _review_path = Path(Y1_RA_SOAT['thu_muc'])
    print('Mã ảnh và tệp nguồn:',_review_path/'anh_can_doi_chieu.csv')
    print('Ghi nhận đã xem:',_review_path/'quyet_dinh_ra_soat.json')
    print('Phương án cập nhật:',_review_path/'phuong_an_sau_ra_soat.csv')
    print('Các tệp này dùng để rà soát và lập phương án; chưa phải bộ huấn luyện đã xuất.')


## 3. Phân tích dữ liệu phát hiện ký tự — YOLO2

**Phần sẽ triển khai sau YOLO1.** Chúng ta phân tích riêng Roboflow OCR v1 và giữ nhãn ký tự gốc để tạo ảnh OCR. Không gộp Winter vào bộ YOLO2.

## 4. Chuẩn bị dữ liệu nhận dạng ký tự — OCR

**Phần sẽ triển khai sau YOLO2.** Chúng ta sẽ thống kê riêng từng nguồn trước khi gộp Roboflow, Winter và CCPD thành train/validation, hướng tới 80/20. Ký tự từ cùng nhóm biển ở cùng một tập. VNLP dành riêng cho test; vẫn kiểm tra trùng giữa các nguồn/tập.

Hướng lọc mới xét từng ký tự: một ký tự đạt cũng có thể được giữ khi đúng vị trí, khớp nhãn và đạt chất lượng. Chưa áp dụng yêu cầu tối thiểu 4–5 ký tự đạt trên một biển. Các ngưỡng chất lượng cần được kiểm chứng bằng một nhóm ảnh mẫu trước khi chạy đầy đủ.

Phạm vi OCR là 32 lớp: `0123456789ABCDEFGHKLMNPQRSTUVXYZ`. Quy mô lấy mẫu sẽ được quyết định sau khi có thống kê; mục tiêu 3k/10k của quy trình cũ không tự chuyển sang bản mới.

## 5. Tổng hợp kết quả và ảnh đại diện

**Phần sẽ hoàn thiện sau các bước xử lý.** Bảng tổng hợp nêu rõ số mẫu ban đầu, số giữ lại và lý do loại. Ảnh đại diện được lưu theo nhiệm vụ → nguồn → kích thước, kèm thông tin để người xem hiểu nhóm ảnh đó đại diện cho phần nào của dữ liệu.

## Nguồn tham khảo về cách trình bày

- Aurélien Géron, *Hands-On Machine Learning*, [notebook chương 2](https://github.com/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb).
- Jeremy Howard và Sylvain Gugger, *Deep Learning for Coders*, [notebook Image Classification](https://github.com/fastai/fastbook/blob/master/05_pet_breeds.ipynb).

Các nguồn được dùng làm mẫu về bố cục giải thích–mã–kết quả; quy trình dữ liệu trong notebook này được viết cho đồ án biển số.
